# 🐘 SQL con PostgreSQL + Python

Notebook de referencia para aprender y repasar SQL desde cero.
Cada sección tiene teoría, ejemplos ejecutables y ejercicios prácticos.

**Motor:** PostgreSQL 16 (Docker)  
**Driver:** psycopg2  
**Schema:** e-commerce (clientes, productos, categorías, pedidos)

---
> ⚠️ **Ejecutar esta primera sección antes de cualquier otra celda**

## 0. Conexión a la base de datos

`psycopg` es el driver oficial de PostgreSQL para Python (v3, la versión moderna).
Antes de ejecutar cualquier query necesitamos importarlo y definir cómo conectarnos.

- `psycopg` — el driver en sí, el que "habla" con PostgreSQL
- `dict_row` — un adaptador que hace que cada fila devuelta sea un diccionario
  (`{"id": 1, "nombre": "Ana"}`) en lugar de una tupla (`(1, "Ana")`)

`CONN_STR` es la cadena de conexión: un string con todos los datos necesarios
para que psycopg encuentre y se autentique con la base de datos.

| Parámetro | Valor | Significado |
|-----------|-------|-------------|
| `host` | `localhost` | La base corre en esta misma máquina (dentro de Docker) |
| `port` | `5433` | Puerto donde Docker expone PostgreSQL |
| `dbname` | `ecommerce` | Nombre de la base de datos a usar |
| `user` | `alumno` | Usuario de PostgreSQL |
| `password` | `alumno123` | Contraseña del usuario |

In [1]:
# sql-aprendizaje/sql_psycopg.ipynb
import psycopg
from psycopg.rows import dict_row

CONN_STR = "host=localhost port=5433 dbname=ecommerce user=alumno password=alumno123"

## Funciones de utilidad

En lugar de repetir el código de conexión en cada celda, definimos dos funciones
reutilizables que usaremos a lo largo de todo el notebook.

### `obtener_conexion()`

Crea y devuelve una conexión a PostgreSQL con dos opciones importantes:

- `autocommit=True` — cada sentencia SQL se confirma automáticamente en la base
  de datos sin necesidad de llamar a `commit()` manualmente. Ideal para un notebook
  de consultas y aprendizaje; en una aplicación real se manejaría con transacciones explícitas.
- `row_factory=dict_row` — le dice a psycopg que devuelva cada fila como diccionario
  en lugar de tupla, así podemos acceder a los datos por nombre de columna: `fila["nombre"]`.

### `ejecutar_query(sql, params=None)`

Ejecuta cualquier sentencia SQL y devuelve los resultados.

- `conn` — la conexión activa a PostgreSQL. El bloque `with` garantiza que se cierre
  automáticamente al terminar, aunque ocurra un error.
- `cur` (cursor) — el objeto que envía las sentencias SQL al servidor y recibe los resultados.
  Análogo a un puntero dentro de la conexión.
- `sql` — la sentencia SQL a ejecutar, como string.
- `params` — valores a insertar en la query de forma segura (evita SQL injection).
  Se pasa como tupla o diccionario y psycopg los escapa automáticamente.
  Ejemplo: `ejecutar_query("SELECT * FROM clientes WHERE id = %s", (1,))`
- `fetchall()` — recupera todas las filas del resultado como lista. Si la sentencia
  no devuelve filas (un `INSERT` o `CREATE TABLE` por ejemplo), lanza una excepción
  que capturamos devolviendo una lista vacía.

In [2]:
def obtener_conexion():
    """Devuelve una conexion con autocommit y filas como diccionarios."""
    return psycopg.connect(CONN_STR, autocommit=True, row_factory=dict_row)

def ejecutar_query(sql, params=None):
    """Ejecuta una query y devuelve los resultados como lista de diccionarios."""
    with obtener_conexion() as conn:
        with conn.cursor() as cur:
            cur.execute(sql, params)
            try:
                return cur.fetchall()
            except Exception:
                return []

### Verificación de la conexión

Antes de continuar, comprobamos que todo el stack está funcionando:
Docker corriendo, PostgreSQL levantado, credenciales correctas.

- `with obtener_conexion() as conn` — abre la conexión y la cierra automáticamente
  al salir del bloque, incluso si ocurre un error.
- `with conn.cursor() as cur` — abre un cursor dentro de esa conexión. Un cursor
  es el canal por donde viajan las sentencias SQL hacia el servidor y los resultados
  de vuelta.
- `cur.execute("SELECT version();")` — ejecuta una query nativa de PostgreSQL que
  devuelve la versión del servidor. Es el "hola mundo" de SQL.
- `fetchone()` — recupera solo la primera fila del resultado. A diferencia de
  `fetchall()`, no trae todo sino una sola fila. Como usamos `dict_row`, esa fila
  es un diccionario.
- `version["version"]` — accedemos al valor por nombre de columna. PostgreSQL
  devuelve la columna con el nombre `version`, entonces usamos esa clave.
- El bloque `try/except` captura cualquier error de conexión (Docker apagado,
  credenciales incorrectas, puerto equivocado) y lo muestra en lugar de romper
  el notebook.

Si la celda imprime la versión de PostgreSQL, el entorno está listo.

In [3]:
try:
    with obtener_conexion() as conn:
        with conn.cursor() as cur:
            cur.execute("SELECT version();")
            version = cur.fetchone()
    print("Conexion exitosa a PostgreSQL")
    print(version["version"])
except Exception as e:
    print(f"Error de conexion: {e}")

Conexion exitosa a PostgreSQL
PostgreSQL 16.14 (Debian 16.14-1.pgdg13+1) on x86_64-pc-linux-gnu, compiled by gcc (Debian 14.2.0-19) 14.2.0, 64-bit


## 01. Fundamentos — DDL, DML y tipos de datos

SQL se divide en tres grandes grupos de sentencias según su propósito:

| Grupo | Nombre completo | Para qué sirve | Ejemplos |
|-------|----------------|----------------|---------|
| DDL | Data Definition Language | Definir la estructura de la base | `CREATE`, `ALTER`, `DROP` |
| DML | Data Manipulation Language | Manipular los datos dentro de las tablas | `INSERT`, `SELECT`, `UPDATE`, `DELETE` |
| DCL | Data Control Language | Controlar permisos y accesos | `GRANT`, `REVOKE` |

En esta sección trabajamos DDL: vamos a crear las tablas del schema e-commerce desde cero.

### Tipos de datos en PostgreSQL

Antes de crear una tabla hay que decidir qué tipo de dato va en cada columna.
Los más usados:

| Tipo | Descripción | Ejemplo |
|------|-------------|---------|
| `SERIAL` | Entero autoincremental, típico para IDs | `1, 2, 3, ...` |
| `INTEGER` | Número entero | `42` |
| `NUMERIC(p, s)` | Decimal de precisión fija. `p` = dígitos totales, `s` = decimales | `NUMERIC(10, 2)` → `12345678.90` |
| `VARCHAR(n)` | Texto de longitud variable con límite `n` | `'Ana García'` |
| `TEXT` | Texto sin límite de longitud | descripción larga |
| `BOOLEAN` | Verdadero o falso | `TRUE`, `FALSE` |
| `DATE` | Fecha sin hora | `2024-01-15` |
| `TIMESTAMP` | Fecha con hora | `2024-01-15 10:30:00` |

`SERIAL` es un shorthand de PostgreSQL: crea un entero con una secuencia automática.
En PostgreSQL moderno también existe `GENERATED ALWAYS AS IDENTITY`, pero `SERIAL`
sigue siendo el más común en la práctica.

### Creando el schema e-commerce

Vamos a crear 4 tablas que representan un sistema de comercio electrónico real.
Por ahora sin constraints entre tablas (eso va en la sección 02), solo la estructura básica.

El orden importa: si una tabla referencia a otra, la referenciada debe existir primero.
En este caso creamos `categorias` antes que `productos` porque productos la va a necesitar.

In [4]:
with obtener_conexion() as conn:
    with conn.cursor() as cur:

        cur.execute("""
            CREATE TABLE IF NOT EXISTS categorias (
                id        SERIAL PRIMARY KEY,
                nombre    VARCHAR(100) NOT NULL,
                activa    BOOLEAN DEFAULT TRUE
            );
        """)

        cur.execute("""
            CREATE TABLE IF NOT EXISTS productos (
                id              SERIAL PRIMARY KEY,
                nombre          VARCHAR(200) NOT NULL,
                precio          NUMERIC(10, 2) NOT NULL,
                stock           INTEGER DEFAULT 0,
                id_categoria    INTEGER,
                creado_en       TIMESTAMP DEFAULT NOW()
            );
        """)

        cur.execute("""
            CREATE TABLE IF NOT EXISTS clientes (
                id          SERIAL PRIMARY KEY,
                nombre      VARCHAR(150) NOT NULL,
                email       VARCHAR(200),
                creado_en   DATE DEFAULT CURRENT_DATE
            );
        """)

        cur.execute("""
            CREATE TABLE IF NOT EXISTS pedidos (
                id          SERIAL PRIMARY KEY,
                id_cliente  INTEGER,
                total       NUMERIC(10, 2),
                estado      VARCHAR(50) DEFAULT 'pendiente',
                creado_en   TIMESTAMP DEFAULT NOW()
            );
        """)

print("Tablas creadas correctamente")

Tablas creadas correctamente


### Verificar las tablas creadas

`information_schema.tables` es una vista del sistema que PostgreSQL mantiene
automáticamente con metadata de todas las tablas existentes.
Filtramos por `table_schema = 'public'` para ver solo las nuestras,
excluyendo las tablas internas del sistema.

In [5]:
tablas = ejecutar_query("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'public'
    ORDER BY table_name;
""")

for tabla in tablas:
    print(tabla["table_name"])

categorias
clientes
pedido_detalle
pedidos
productos


## 02. Constraints — Restricciones de integridad

Un constraint es una regla que PostgreSQL aplica automáticamente sobre los datos.
Si una operación viola una regla, el motor la rechaza y lanza un error.
Sirven para garantizar que la base de datos nunca tenga datos inválidos,
sin depender de que la aplicación lo valide correctamente.

| Constraint | Para qué sirve |
|------------|----------------|
| `PRIMARY KEY` | Identifica de forma única cada fila. Implica `NOT NULL` + `UNIQUE` |
| `FOREIGN KEY` | Vincula una columna con la PK de otra tabla. Garantiza integridad referencial |
| `UNIQUE` | No permite valores duplicados en esa columna |
| `NOT NULL` | La columna no puede quedar vacía |
| `DEFAULT` | Valor que toma la columna si no se especifica uno al insertar |
| `CHECK` | Valida que el valor cumpla una condición lógica arbitraria |

### Recreando el schema con constraints completos

En la sección anterior creamos las tablas sin relaciones entre ellas.
Ahora las eliminamos y las volvemos a crear con todos los constraints.

`DROP TABLE IF EXISTS ... CASCADE` elimina la tabla y cualquier Foreign Key
que otras tablas tengan hacia ella, evitando errores de dependencia.
El orden de eliminación es el inverso al de creación: primero las que
dependen de otras, después las independientes.

In [6]:
with obtener_conexion() as conn:
    with conn.cursor() as cur:
        cur.execute("""
            DROP TABLE IF EXISTS pedidos   CASCADE;
            DROP TABLE IF EXISTS productos CASCADE;
            DROP TABLE IF EXISTS clientes  CASCADE;
            DROP TABLE IF EXISTS categorias CASCADE;
        """)

print("Tablas eliminadas")

Tablas eliminadas


### PRIMARY KEY y NOT NULL

`PRIMARY KEY` es el identificador único de cada fila.
- Solo puede haber una por tabla
- No acepta `NULL`
- PostgreSQL crea automáticamente un índice sobre ella (las búsquedas por ID son rápidas)

`NOT NULL` garantiza que la columna siempre tenga un valor.
Sin este constraint, cualquier columna acepta `NULL` por defecto.

In [7]:
with obtener_conexion() as conn:
    with conn.cursor() as cur:
        cur.execute("""
            CREATE TABLE categorias (
                id      SERIAL PRIMARY KEY,
                nombre  VARCHAR(100) NOT NULL,
                activa  BOOLEAN DEFAULT TRUE
            );
        """)

print("Tabla categorias creada")

Tabla categorias creada


### FOREIGN KEY

Una Foreign Key (FK) vincula una columna de una tabla con la Primary Key de otra.
Garantiza integridad referencial: no se puede insertar un `id_categoria` en `productos`
que no exista en `categorias`.

`ON DELETE` define qué pasa con los productos si se elimina su categoría:
- `RESTRICT` — rechaza el DELETE si hay productos que la referencian (la más segura)
- `CASCADE` — elimina también todos los productos de esa categoría
- `SET NULL` — pone `NULL` en `id_categoria` de los productos huérfanos

In [8]:
with obtener_conexion() as conn:
    with conn.cursor() as cur:
        cur.execute("""
            CREATE TABLE productos (
                id            SERIAL PRIMARY KEY,
                nombre        VARCHAR(200) NOT NULL,
                precio        NUMERIC(10, 2) NOT NULL,
                stock         INTEGER DEFAULT 0,
                id_categoria  INTEGER REFERENCES categorias(id) ON DELETE RESTRICT,
                creado_en     TIMESTAMP DEFAULT NOW()
            );
        """)

print("Tabla productos creada")

Tabla productos creada


### UNIQUE y CHECK

`UNIQUE` impide duplicados en una columna. A diferencia de `PRIMARY KEY`,
puede haber varias columnas `UNIQUE` en una tabla, y sí acepta `NULL`
(PostgreSQL considera que dos `NULL` no son iguales entre sí).

`CHECK` valida una condición arbitraria antes de aceptar el valor.
Si la condición devuelve `FALSE`, PostgreSQL rechaza la operación.
Ejemplos útiles:
- Que el precio no sea negativo
- Que el stock no baje de cero  
- Que el estado solo pueda ser uno de un conjunto de valores

In [9]:
with obtener_conexion() as conn:
    with conn.cursor() as cur:
        cur.execute("""
            CREATE TABLE clientes (
                id         SERIAL PRIMARY KEY,
                nombre     VARCHAR(150) NOT NULL,
                email      VARCHAR(200) UNIQUE,
                creado_en  DATE DEFAULT CURRENT_DATE
            );
        """)

        cur.execute("""
            CREATE TABLE pedidos (
                id          SERIAL PRIMARY KEY,
                id_cliente  INTEGER REFERENCES clientes(id) ON DELETE RESTRICT,
                total       NUMERIC(10, 2) CHECK (total >= 0),
                estado      VARCHAR(50) DEFAULT 'pendiente'
                                CHECK (estado IN ('pendiente', 'pagado', 'enviado', 'cancelado')),
                creado_en   TIMESTAMP DEFAULT NOW()
            );
        """)

print("Tablas clientes y pedidos creadas")

Tablas clientes y pedidos creadas


### Verificar los constraints creados

`information_schema.table_constraints` lista todos los constraints de la base.
Filtramos por nuestras cuatro tablas para ver qué reglas quedaron activas.

In [10]:
constraints = ejecutar_query("""
    SELECT table_name, constraint_name, constraint_type
    FROM information_schema.table_constraints
    WHERE table_schema = 'public'
    ORDER BY table_name, constraint_type;
""")

for c in constraints:
    print(f"{c['table_name']:15} {c['constraint_type']:12} {c['constraint_name']}")

categorias      CHECK        2200_17424_1_not_null
categorias      CHECK        2200_17424_2_not_null
categorias      PRIMARY KEY  categorias_pkey
clientes        CHECK        2200_17446_1_not_null
clientes        CHECK        2200_17446_2_not_null
clientes        PRIMARY KEY  clientes_pkey
clientes        UNIQUE       clientes_email_key
pedido_detalle  CHECK        2200_16688_1_not_null
pedido_detalle  CHECK        pedido_detalle_cantidad_check
pedido_detalle  CHECK        2200_16688_4_not_null
pedido_detalle  CHECK        2200_16688_5_not_null
pedido_detalle  PRIMARY KEY  pedido_detalle_pkey
pedidos         CHECK        pedidos_estado_check
pedidos         CHECK        pedidos_total_check
pedidos         CHECK        2200_17456_1_not_null
pedidos         FOREIGN KEY  pedidos_id_cliente_fkey
pedidos         PRIMARY KEY  pedidos_pkey
productos       CHECK        2200_17432_2_not_null
productos       CHECK        2200_17432_1_not_null
productos       CHECK        2200_17432_3_not_null
p

## 03. CRUD — INSERT, SELECT, UPDATE, DELETE

CRUD es el acrónimo de las cuatro operaciones básicas sobre datos:

| Operación | SQL | Descripción |
|-----------|-----|-------------|
| Create | `INSERT` | Agregar filas nuevas a una tabla |
| Read | `SELECT` | Consultar y recuperar datos |
| Update | `UPDATE` | Modificar filas existentes |
| Delete | `DELETE` | Eliminar filas |

Todas pertenecen al grupo DML (Data Manipulation Language).
A partir de acá empezamos a poblar el schema e-commerce con datos reales
y a consultarlos de distintas formas.

### INSERT — Agregar datos

Sintaxis básica:
```sql
INSERT INTO tabla (columna1, columna2) VALUES (valor1, valor2);
```

Puntos importantes:
- Las columnas con `DEFAULT` o `SERIAL` pueden omitirse; PostgreSQL las completa solo.
- Para insertar múltiples filas en una sola sentencia se encadenan los `VALUES`.
- psycopg usa `%s` como placeholder para parámetros seguros — nunca se concatena
  el valor directamente en el string SQL, eso abre vulnerabilidades de SQL injection.

In [11]:
with obtener_conexion() as conn:
    with conn.cursor() as cur:

        # Insertar categorias
        cur.executemany(
            "INSERT INTO categorias (nombre) VALUES (%s)",
            [("Electronica",), ("Ropa",), ("Hogar",), ("Deportes",)]
        )

        # Insertar productos con parametros nombrados
        productos = [
            ("Laptop 15\"",   1299.99, 10, 1),
            ("Auriculares",    89.99, 35, 1),
            ("Camiseta M",     19.99, 100, 2),
            ("Zapatillas 42",  79.99, 50, 4),
            ("Lampara LED",    34.99, 60, 3),
        ]
        cur.executemany(
            "INSERT INTO productos (nombre, precio, stock, id_categoria) VALUES (%s, %s, %s, %s)",
            productos
        )

        # Insertar clientes
        clientes = [
            ("Ana Torres",    "ana@email.com"),
            ("Luis Gomez",    "luis@email.com"),
            ("Maria Lopez",   "maria@email.com"),
            ("Carlos Ruiz",   "carlos@email.com"),
        ]
        cur.executemany(
            "INSERT INTO clientes (nombre, email) VALUES (%s, %s)",
            clientes
        )

        # Insertar pedidos
        pedidos = [
            (1, 1389.98, "pagado"),
            (2,   79.99, "enviado"),
            (1,   34.99, "pendiente"),
            (3,   89.99, "pagado"),
        ]
        cur.executemany(
            "INSERT INTO pedidos (id_cliente, total, estado) VALUES (%s, %s, %s)",
            pedidos
        )

print("Datos insertados correctamente")

Datos insertados correctamente


### SELECT — Consultar datos

`SELECT` es la sentencia más usada en SQL. Su estructura básica:

```sql
SELECT columnas
FROM tabla
WHERE condicion
ORDER BY columna;
```

- `*` selecciona todas las columnas. En producción conviene listar solo las necesarias.
- `WHERE` filtra filas — sin él se devuelven todas.
- `ORDER BY` ordena el resultado. `ASC` es ascendente (default), `DESC` descendente.
- `LIMIT` restringe la cantidad de filas devueltas, útil para explorar tablas grandes.

In [12]:
# Todos los productos
productos = ejecutar_query("SELECT * FROM productos ORDER BY precio DESC;")
for p in productos:
    print(f"{p['nombre']:20} ${p['precio']}")

print()

# Solo nombre y email de clientes, ordenados alfabeticamente
clientes = ejecutar_query("SELECT nombre, email FROM clientes ORDER BY nombre ASC;")
for c in clientes:
    print(f"{c['nombre']:20} {c['email']}")

print()

# Los 3 productos mas baratos
baratos = ejecutar_query("SELECT nombre, precio FROM productos ORDER BY precio ASC LIMIT 3;")
for p in baratos:
    print(f"{p['nombre']:20} ${p['precio']}")

Laptop 15"           $1299.99
Auriculares          $89.99
Zapatillas 42        $79.99
Lampara LED          $34.99
Camiseta M           $19.99

Ana Torres           ana@email.com
Carlos Ruiz          carlos@email.com
Luis Gomez           luis@email.com
Maria Lopez          maria@email.com

Camiseta M           $19.99
Lampara LED          $34.99
Zapatillas 42        $79.99


### UPDATE — Modificar datos

```sql
UPDATE tabla
SET columna1 = valor1, columna2 = valor2
WHERE condicion;
```

⚠️ El `WHERE` es crítico: sin él se actualizan **todas** las filas de la tabla.
Siempre verificar la condición antes de ejecutar.

`RETURNING` es una extensión de PostgreSQL que devuelve las filas afectadas
después del UPDATE, útil para confirmar qué cambió sin hacer una query adicional.

In [13]:
with obtener_conexion() as conn:
    with conn.cursor() as cur:

        # Actualizar precio de un producto
        cur.execute("""
            UPDATE productos
            SET precio = 1199.99
            WHERE nombre = 'Laptop 15"'
            RETURNING id, nombre, precio;
        """)
        actualizado = cur.fetchone()
        print(f"Actualizado: {actualizado['nombre']} → ${actualizado['precio']}")

        # Cambiar estado de un pedido
        cur.execute("""
            UPDATE pedidos
            SET estado = 'enviado'
            WHERE id = 3
            RETURNING id, estado;
        """)
        pedido = cur.fetchone()
        print(f"Pedido {pedido['id']} → {pedido['estado']}")

Actualizado: Laptop 15" → $1199.99
Pedido 3 → enviado


### DELETE — Eliminar datos

```sql
DELETE FROM tabla
WHERE condicion;
```

⚠️ Igual que `UPDATE`, sin `WHERE` elimina todas las filas.
A diferencia de `DROP TABLE`, `DELETE` elimina filas pero mantiene la tabla.

Alternativa para vaciar una tabla completamente: `TRUNCATE tabla` —
más rápido que `DELETE` sin `WHERE` porque no recorre fila por fila,
pero no dispara triggers y no se puede filtrar.

In [14]:
with obtener_conexion() as conn:
    with conn.cursor() as cur:

        # Insertar un cliente de prueba para luego eliminarlo
        cur.execute("""
            INSERT INTO clientes (nombre, email)
            VALUES ('Prueba Borrar', 'borrar@email.com')
            RETURNING id, nombre;
        """)
        nuevo = cur.fetchone()
        print(f"Insertado: id={nuevo['id']} nombre={nuevo['nombre']}")

        # Eliminar ese cliente
        cur.execute("""
            DELETE FROM clientes
            WHERE email = 'borrar@email.com'
            RETURNING id, nombre;
        """)
        eliminado = cur.fetchone()
        print(f"Eliminado: id={eliminado['id']} nombre={eliminado['nombre']}")

# Verificar que solo quedan los 4 clientes originales
clientes = ejecutar_query("SELECT COUNT(*) as total FROM clientes;")
print(f"Clientes en tabla: {clientes[0]['total']}")

Insertado: id=5 nombre=Prueba Borrar
Eliminado: id=5 nombre=Prueba Borrar
Clientes en tabla: 4


## 04. Filtros — WHERE, BETWEEN, IN, LIKE, IS NULL

`WHERE` es la cláusula que filtra qué filas devuelve o afecta una sentencia.
Sin ella, `SELECT`, `UPDATE` y `DELETE` operan sobre toda la tabla.

Los filtros se construyen con operadores que se pueden combinar con `AND` y `OR`:

| Operador | Descripción | Ejemplo |
|----------|-------------|---------|
| `=` | Igual | `estado = 'pagado'` |
| `<>` o `!=` | Distinto | `estado <> 'cancelado'` |
| `>` `<` `>=` `<=` | Comparación numérica o de fechas | `precio > 100` |
| `BETWEEN` | Rango inclusivo en ambos extremos | `precio BETWEEN 50 AND 200` |
| `IN` | Pertenece a un conjunto de valores | `estado IN ('pagado', 'enviado')` |
| `LIKE` | Coincidencia de patrón en texto | `nombre LIKE 'La%'` |
| `IS NULL` | La columna no tiene valor | `email IS NULL` |
| `IS NOT NULL` | La columna tiene valor | `email IS NOT NULL` |

### Operadores de comparación y lógicos

`AND` requiere que todas las condiciones sean verdaderas.
`OR` requiere que al menos una lo sea.
`NOT` invierte el resultado de una condición.

El orden de precedencia es `NOT` → `AND` → `OR`.
Cuando se mezclan, usar paréntesis para dejar la intención explícita.

In [15]:
# Productos con precio mayor a 50
print("--- Precio > 50 ---")
resultado = ejecutar_query("""
    SELECT nombre, precio
    FROM productos
    WHERE precio > 50
    ORDER BY precio;
""")
for r in resultado:
    print(f"{r['nombre']:20} ${r['precio']}")

# Productos baratos con stock alto
print("\n--- Precio < 50 AND stock > 50 ---")
resultado = ejecutar_query("""
    SELECT nombre, precio, stock
    FROM productos
    WHERE precio < 50 AND stock > 50
    ORDER BY stock DESC;
""")
for r in resultado:
    print(f"{r['nombre']:20} ${r['precio']} stock:{r['stock']}")

--- Precio > 50 ---
Zapatillas 42        $79.99
Auriculares          $89.99
Laptop 15"           $1199.99

--- Precio < 50 AND stock > 50 ---
Camiseta M           $19.99 stock:100
Lampara LED          $34.99 stock:60


### BETWEEN — Rango de valores

```sql
WHERE columna BETWEEN valor_minimo AND valor_maximo
```

Equivale a `columna >= minimo AND columna <= maximo` — ambos extremos son inclusivos.
Funciona con números, fechas y texto (orden alfabético).

`NOT BETWEEN` devuelve los valores fuera del rango.

In [16]:
# Productos en rango de precio
print("--- Precio BETWEEN 30 AND 100 ---")
resultado = ejecutar_query("""
    SELECT nombre, precio
    FROM productos
    WHERE precio BETWEEN 30 AND 100
    ORDER BY precio;
""")
for r in resultado:
    print(f"{r['nombre']:20} ${r['precio']}")

# Pedidos fuera de ese rango
print("\n--- Pedidos NOT BETWEEN 50 AND 500 ---")
resultado = ejecutar_query("""
    SELECT id, total, estado
    FROM pedidos
    WHERE total NOT BETWEEN 50 AND 500
    ORDER BY total;
""")
for r in resultado:
    print(f"Pedido {r['id']}  ${r['total']}  {r['estado']}")

--- Precio BETWEEN 30 AND 100 ---


Lampara LED          $34.99
Zapatillas 42        $79.99
Auriculares          $89.99

--- Pedidos NOT BETWEEN 50 AND 500 ---
Pedido 3  $34.99  enviado
Pedido 1  $1389.98  pagado


### IN — Conjunto de valores

```sql
WHERE columna IN (valor1, valor2, valor3)
```

Es un shorthand legible de múltiples condiciones `OR`.
Esta query:
```sql
WHERE estado IN ('pagado', 'enviado')
```
es equivalente a:
```sql
WHERE estado = 'pagado' OR estado = 'enviado'
```

`NOT IN` excluye los valores del conjunto.
Cuidado con `NOT IN` cuando el conjunto puede contener `NULL` —
en ese caso la condición entera devuelve `NULL` y no filtra como se espera.

In [17]:
# Pedidos pagados o enviados
print("--- Estado IN ('pagado', 'enviado') ---")
resultado = ejecutar_query("""
    SELECT id, total, estado
    FROM pedidos
    WHERE estado IN ('pagado', 'enviado')
    ORDER BY estado;
""")
for r in resultado:
    print(f"Pedido {r['id']}  ${r['total']}  {r['estado']}")

# Productos de categorias especificas
print("\n--- Categorias 1 y 4 ---")
resultado = ejecutar_query("""
    SELECT nombre, id_categoria
    FROM productos
    WHERE id_categoria IN (1, 4)
    ORDER BY id_categoria;
""")
for r in resultado:
    print(f"{r['nombre']:20} cat:{r['id_categoria']}")

--- Estado IN ('pagado', 'enviado') ---
Pedido 2  $79.99  enviado
Pedido 3  $34.99  enviado
Pedido 1  $1389.98  pagado
Pedido 4  $89.99  pagado

--- Categorias 1 y 4 ---
Auriculares          cat:1
Laptop 15"           cat:1
Zapatillas 42        cat:4


### LIKE — Coincidencia de patrones

```sql
WHERE columna LIKE 'patron'
```

Los caracteres especiales del patrón son:
- `%` — cualquier secuencia de caracteres (incluyendo ninguno)
- `_` — exactamente un carácter cualquiera

Ejemplos:
| Patrón | Coincide con |
|--------|-------------|
| `'La%'` | Todo lo que empieza con "La" |
| `'%LED'` | Todo lo que termina en "LED" |
| `'%a%'` | Todo lo que contiene "a" |
| `'_a%'` | Todo lo que tiene "a" en la segunda posición |

`ILIKE` es la versión case-insensitive de PostgreSQL (no existe en todos los motores SQL).

In [18]:
# Productos que empiezan con 'La'
print("--- LIKE 'La%' ---")
resultado = ejecutar_query("""
    SELECT nombre FROM productos
    WHERE nombre LIKE 'La%';
""")
for r in resultado:
    print(r['nombre'])

# Clientes cuyo email termina en .com
print("\n--- Email LIKE '%.com' ---")
resultado = ejecutar_query("""
    SELECT nombre, email FROM clientes
    WHERE email LIKE '%.com'
    ORDER BY nombre;
""")
for r in resultado:
    print(f"{r['nombre']:20} {r['email']}")

# Busqueda case-insensitive
print("\n--- Nombre ILIKE '%lopez%' ---")
resultado = ejecutar_query("""
    SELECT nombre FROM clientes
    WHERE nombre ILIKE '%lopez%';
""")
for r in resultado:
    print(r['nombre'])

--- LIKE 'La%' ---
Lampara LED
Laptop 15"

--- Email LIKE '%.com' ---
Ana Torres           ana@email.com
Carlos Ruiz          carlos@email.com
Luis Gomez           luis@email.com
Maria Lopez          maria@email.com

--- Nombre ILIKE '%lopez%' ---
Maria Lopez


### IS NULL / IS NOT NULL

`NULL` en SQL significa ausencia de valor — no es un string vacío ni un cero.
Por eso no se puede comparar con `=`: `columna = NULL` siempre devuelve `NULL`,
nunca `TRUE`. La única forma correcta es `IS NULL` o `IS NOT NULL`.

Casos típicos:
- Clientes que no proporcionaron email
- Productos sin categoría asignada
- Pedidos sin fecha de envío registrada

In [19]:
# Insertar un cliente sin email para el ejemplo
with obtener_conexion() as conn:
    with conn.cursor() as cur:
        cur.execute("""
            INSERT INTO clientes (nombre)
            VALUES ('Sin Email')
            RETURNING id, nombre, email;
        """)
        nuevo = cur.fetchone()
        print(f"Insertado: {nuevo['nombre']} — email: {nuevo['email']}")

# Clientes sin email
print("\n--- Email IS NULL ---")
resultado = ejecutar_query("""
    SELECT nombre, email FROM clientes
    WHERE email IS NULL;
""")
for r in resultado:
    print(f"{r['nombre']} — sin email")

# Clientes con email
print("\n--- Email IS NOT NULL ---")
resultado = ejecutar_query("""
    SELECT nombre, email FROM clientes
    WHERE email IS NOT NULL
    ORDER BY nombre;
""")
for r in resultado:
    print(f"{r['nombre']:20} {r['email']}")

Insertado: Sin Email — email: None

--- Email IS NULL ---
Sin Email — sin email

--- Email IS NOT NULL ---
Ana Torres           ana@email.com
Carlos Ruiz          carlos@email.com
Luis Gomez           luis@email.com
Maria Lopez          maria@email.com


## 05. Agregación — GROUP BY, HAVING, COUNT, SUM, AVG, MIN, MAX

Las funciones de agregado calculan un resultado sobre un conjunto de filas
y devuelven un único valor. Se usan para responder preguntas como
"¿cuántos?", "¿cuánto en total?", "¿cuál es el promedio?".

| Función | Descripción |
|---------|-------------|
| `COUNT(*)` | Cuenta todas las filas del grupo |
| `COUNT(columna)` | Cuenta filas donde esa columna no es `NULL` |
| `SUM(columna)` | Suma los valores |
| `AVG(columna)` | Promedio aritmético |
| `MIN(columna)` | Valor mínimo |
| `MAX(columna)` | Valor máximo |

Estas funciones por sí solas operan sobre toda la tabla.
`GROUP BY` las hace operar sobre subgrupos de filas.

### Funciones de agregado sobre toda la tabla

Sin `GROUP BY`, la función agrega todas las filas y devuelve un único resultado.
Útil para métricas globales: total de productos, precio promedio, etc.

In [20]:
# Metricas globales de productos
resultado = ejecutar_query("""
    SELECT
        COUNT(*)            AS total_productos,
        ROUND(AVG(precio), 2) AS precio_promedio,
        MIN(precio)         AS precio_minimo,
        MAX(precio)         AS precio_maximo,
        SUM(stock)          AS stock_total
    FROM productos;
""")
r = resultado[0]
print(f"Total productos : {r['total_productos']}")
print(f"Precio promedio : ${r['precio_promedio']}")
print(f"Precio minimo   : ${r['precio_minimo']}")
print(f"Precio maximo   : ${r['precio_maximo']}")
print(f"Stock total     : {r['stock_total']} unidades")

Total productos : 5
Precio promedio : $284.99
Precio minimo   : $19.99
Precio maximo   : $1199.99
Stock total     : 255 unidades


### GROUP BY — Agrupar por categoría

```sql
SELECT columna, FUNCION(otra_columna)
FROM tabla
GROUP BY columna;
```

`GROUP BY` divide las filas en grupos según los valores de una columna
y aplica la función de agregado a cada grupo por separado.

Regla fundamental: toda columna en el `SELECT` que no sea una función
de agregado **debe** aparecer en el `GROUP BY`. Si no, PostgreSQL
lanza un error porque no sabe qué valor mostrar para esa columna.

In [21]:
# Cantidad de productos por categoria
print("--- Productos por categoria ---")
resultado = ejecutar_query("""
    SELECT
        c.nombre            AS categoria,
        COUNT(p.id)         AS cantidad,
        ROUND(AVG(p.precio), 2) AS precio_promedio
    FROM categorias c
    LEFT JOIN productos p ON p.id_categoria = c.id
    GROUP BY c.id, c.nombre
    ORDER BY cantidad DESC;
""")
for r in resultado:
    print(f"{r['categoria']:15} {r['cantidad']} productos  avg: ${r['precio_promedio']}")

print()

# Total facturado por estado de pedido
print("--- Total por estado de pedido ---")
resultado = ejecutar_query("""
    SELECT
        estado,
        COUNT(*)        AS cantidad,
        SUM(total)      AS facturado
    FROM pedidos
    GROUP BY estado
    ORDER BY facturado DESC;
""")
for r in resultado:
    print(f"{r['estado']:12} {r['cantidad']} pedidos  total: ${r['facturado']}")

--- Productos por categoria ---
Electronica     2 productos  avg: $644.99
Deportes        1 productos  avg: $79.99
Ropa            1 productos  avg: $19.99
Hogar           1 productos  avg: $34.99

--- Total por estado de pedido ---
pagado       2 pedidos  total: $1479.97
enviado      2 pedidos  total: $114.98


### HAVING — Filtrar grupos

`WHERE` filtra filas individuales **antes** de agrupar.
`HAVING` filtra grupos **después** de aplicar el agregado.

```sql
SELECT columna, COUNT(*)
FROM tabla
GROUP BY columna
HAVING COUNT(*) > 2;
```

No se puede usar una función de agregado en `WHERE` — para eso existe `HAVING`.
El orden de ejecución en SQL es:
`FROM` → `WHERE` → `GROUP BY` → `HAVING` → `SELECT` → `ORDER BY`

In [22]:
# Categorias con mas de 1 producto
print("--- Categorias con mas de 1 producto ---")
resultado = ejecutar_query("""
    SELECT
        c.nombre        AS categoria,
        COUNT(p.id)     AS cantidad
    FROM categorias c
    LEFT JOIN productos p ON p.id_categoria = c.id
    GROUP BY c.id, c.nombre
    HAVING COUNT(p.id) > 1
    ORDER BY cantidad DESC;
""")
for r in resultado:
    print(f"{r['categoria']:15} {r['cantidad']} productos")

print()

# Clientes con mas de 1 pedido
print("--- Clientes con mas de 1 pedido ---")
resultado = ejecutar_query("""
    SELECT
        id_cliente,
        COUNT(*)    AS total_pedidos,
        SUM(total)  AS total_gastado
    FROM pedidos
    GROUP BY id_cliente
    HAVING COUNT(*) > 1
    ORDER BY total_gastado DESC;
""")
for r in resultado:
    print(f"Cliente {r['id_cliente']}  {r['total_pedidos']} pedidos  ${r['total_gastado']}")

--- Categorias con mas de 1 producto ---
Electronica     2 productos

--- Clientes con mas de 1 pedido ---
Cliente 1  2 pedidos  $1424.97


### WHERE + GROUP BY + HAVING juntos

Los tres se pueden combinar en una misma query.
Recordar el orden de ejecución:

1. `FROM` — qué tabla
2. `WHERE` — filtrar filas individuales
3. `GROUP BY` — agrupar las filas restantes
4. `HAVING` — filtrar los grupos
5. `SELECT` — calcular las columnas del resultado
6. `ORDER BY` — ordenar el resultado final

En el ejemplo siguiente: primero descartamos los pedidos cancelados (`WHERE`),
luego agrupamos por cliente (`GROUP BY`), y finalmente nos quedamos
solo con los que gastaron más de $100 (`HAVING`).

In [23]:
resultado = ejecutar_query("""
    SELECT
        id_cliente,
        COUNT(*)            AS pedidos,
        SUM(total)          AS total_gastado,
        ROUND(AVG(total), 2) AS ticket_promedio
    FROM pedidos
    WHERE estado <> 'cancelado'
    GROUP BY id_cliente
    HAVING SUM(total) > 100
    ORDER BY total_gastado DESC;
""")

print("--- Clientes con mas de $100 en pedidos no cancelados ---")
for r in resultado:
    print(f"Cliente {r['id_cliente']}  {r['pedidos']} pedidos  "
          f"total: ${r['total_gastado']}  avg: ${r['ticket_promedio']}")

--- Clientes con mas de $100 en pedidos no cancelados ---
Cliente 1  2 pedidos  total: $1424.97  avg: $712.49


## 06. JOINs — INNER, LEFT, RIGHT, FULL, SELF, CROSS

Un JOIN combina filas de dos o más tablas basándose en una condición,
generalmente la relación entre una Foreign Key y una Primary Key.

Sin JOINs estaríamos limitados a consultar una tabla a la vez.
Con JOINs podemos responder preguntas como "¿qué productos pidió cada cliente?"
o "¿a qué categoría pertenece cada producto?".

| Tipo | Devuelve |
|------|----------|
| `INNER JOIN` | Solo filas que tienen coincidencia en ambas tablas |
| `LEFT JOIN` | Todas las filas de la tabla izquierda, coincidan o no |
| `RIGHT JOIN` | Todas las filas de la tabla derecha, coincidan o no |
| `FULL JOIN` | Todas las filas de ambas tablas, coincidan o no |
| `SELF JOIN` | Una tabla unida consigo misma |
| `CROSS JOIN` | Producto cartesiano — todas las combinaciones posibles |

La tabla "izquierda" es la que aparece antes del JOIN en la query.
La tabla "derecha" es la que aparece después.

### INNER JOIN

Devuelve únicamente las filas que tienen coincidencia en ambas tablas.
Si un producto no tiene categoría asignada, no aparece en el resultado.
Si una categoría no tiene productos, tampoco aparece.

```sql
SELECT *
FROM tabla_izquierda
INNER JOIN tabla_derecha ON tabla_izquierda.columna = tabla_derecha.columna;
```

`INNER` es opcional — `JOIN` a secas equivale a `INNER JOIN`.

In [24]:
# Productos con su categoria
print("--- Productos con categoria (INNER JOIN) ---")
resultado = ejecutar_query("""
    SELECT
        p.nombre        AS producto,
        p.precio,
        c.nombre        AS categoria
    FROM productos p
    INNER JOIN categorias c ON p.id_categoria = c.id
    ORDER BY c.nombre, p.precio;
""")
for r in resultado:
    print(f"{r['producto']:20} ${r['precio']:8}  [{r['categoria']}]")

print()

# Pedidos con datos del cliente
print("--- Pedidos con cliente (INNER JOIN) ---")
resultado = ejecutar_query("""
    SELECT
        p.id            AS pedido_id,
        c.nombre        AS cliente,
        p.total,
        p.estado
    FROM pedidos p
    INNER JOIN clientes c ON p.id_cliente = c.id
    ORDER BY p.id;
""")
for r in resultado:
    print(f"Pedido {r['pedido_id']}  {r['cliente']:15} ${r['total']}  {r['estado']}")

--- Productos con categoria (INNER JOIN) ---
Zapatillas 42        $   79.99  [Deportes]
Auriculares          $   89.99  [Electronica]
Laptop 15"           $ 1199.99  [Electronica]
Lampara LED          $   34.99  [Hogar]
Camiseta M           $   19.99  [Ropa]

--- Pedidos con cliente (INNER JOIN) ---
Pedido 1  Ana Torres      $1389.98  pagado
Pedido 2  Luis Gomez      $79.99  enviado
Pedido 3  Ana Torres      $34.99  enviado
Pedido 4  Maria Lopez     $89.99  pagado


### LEFT JOIN

Devuelve todas las filas de la tabla izquierda.
Si no hay coincidencia en la tabla derecha, las columnas de esa tabla
aparecen como `NULL`.

Es el JOIN más usado en la práctica porque permite ver registros
"huérfanos" — por ejemplo, categorías sin productos o clientes sin pedidos.

```sql
SELECT *
FROM tabla_izquierda
LEFT JOIN tabla_derecha ON condicion;
```

In [25]:
# Todas las categorias, tengan o no productos
print("--- Categorias con y sin productos (LEFT JOIN) ---")
resultado = ejecutar_query("""
    SELECT
        c.nombre        AS categoria,
        COUNT(p.id)     AS cantidad_productos
    FROM categorias c
    LEFT JOIN productos p ON p.id_categoria = c.id
    GROUP BY c.id, c.nombre
    ORDER BY cantidad_productos DESC;
""")
for r in resultado:
    print(f"{r['categoria']:15} {r['cantidad_productos']} productos")

print()

# Clientes con y sin pedidos
print("--- Clientes con y sin pedidos (LEFT JOIN) ---")
resultado = ejecutar_query("""
    SELECT
        c.nombre        AS cliente,
        p.id            AS pedido_id,
        p.total,
        p.estado
    FROM clientes c
    LEFT JOIN pedidos p ON p.id_cliente = c.id
    ORDER BY c.nombre;
""")
for r in resultado:
    pedido = f"Pedido {r['pedido_id']}  ${r['total']}  {r['estado']}" if r['pedido_id'] else "sin pedidos"
    print(f"{r['cliente']:20} {pedido}")

--- Categorias con y sin productos (LEFT JOIN) ---
Electronica     2 productos
Deportes        1 productos
Ropa            1 productos
Hogar           1 productos

--- Clientes con y sin pedidos (LEFT JOIN) ---
Ana Torres           Pedido 1  $1389.98  pagado
Ana Torres           Pedido 3  $34.99  enviado
Carlos Ruiz          sin pedidos
Luis Gomez           Pedido 2  $79.99  enviado
Maria Lopez          Pedido 4  $89.99  pagado
Sin Email            sin pedidos


### RIGHT JOIN

Espejo del `LEFT JOIN`: devuelve todas las filas de la tabla derecha,
con `NULL` en las columnas de la izquierda cuando no hay coincidencia.

En la práctica se usa poco — cualquier `RIGHT JOIN` se puede reescribir
como un `LEFT JOIN` invirtiendo el orden de las tablas, que resulta más legible.

```sql
-- Estas dos queries son equivalentes:
SELECT * FROM a RIGHT JOIN b ON a.id = b.id_a;
SELECT * FROM b LEFT JOIN a ON a.id = b.id_a;
```

In [26]:
# Todos los productos, esten o no en un pedido
# (RIGHT JOIN visto desde pedidos hacia productos)
print("--- Productos referenciados en pedidos (RIGHT JOIN) ---")
resultado = ejecutar_query("""
    SELECT
        p.nombre        AS producto,
        p.precio,
        c.nombre        AS categoria
    FROM categorias c
    RIGHT JOIN productos p ON p.id_categoria = c.id
    ORDER BY p.nombre;
""")
for r in resultado:
    cat = r['categoria'] if r['categoria'] else 'sin categoria'
    print(f"{r['producto']:20} ${r['precio']}  [{cat}]")

--- Productos referenciados en pedidos (RIGHT JOIN) ---
Auriculares          $89.99  [Electronica]
Camiseta M           $19.99  [Ropa]
Lampara LED          $34.99  [Hogar]
Laptop 15"           $1199.99  [Electronica]
Zapatillas 42        $79.99  [Deportes]


### FULL JOIN

Combina `LEFT JOIN` y `RIGHT JOIN`: devuelve todas las filas de ambas tablas.
Donde no hay coincidencia, las columnas de la tabla sin match aparecen como `NULL`.

Útil para detectar inconsistencias: registros huérfanos en cualquiera de los dos lados.

```sql
SELECT *
FROM tabla_izquierda
FULL JOIN tabla_derecha ON condicion;
```

In [27]:
# Ver todos los clientes y todos los pedidos, haya o no relacion
print("--- FULL JOIN clientes y pedidos ---")
resultado = ejecutar_query("""
    SELECT
        c.nombre        AS cliente,
        p.id            AS pedido_id,
        p.total,
        p.estado
    FROM clientes c
    FULL JOIN pedidos p ON p.id_cliente = c.id
    ORDER BY c.nombre NULLS LAST;
""")
for r in resultado:
    cliente = r['cliente'] if r['cliente'] else 'sin cliente'
    pedido  = f"Pedido {r['pedido_id']}  ${r['total']}  {r['estado']}" if r['pedido_id'] else 'sin pedidos'
    print(f"{cliente:20} {pedido}")

--- FULL JOIN clientes y pedidos ---
Ana Torres           Pedido 1  $1389.98  pagado
Ana Torres           Pedido 3  $34.99  enviado
Carlos Ruiz          sin pedidos
Luis Gomez           Pedido 2  $79.99  enviado
Maria Lopez          Pedido 4  $89.99  pagado
Sin Email            sin pedidos


### SELF JOIN

Una tabla unida consigo misma. No existe una sintaxis especial —
es un JOIN normal donde las dos tablas son la misma, usando aliases
distintos para distinguirlas.

Caso de uso clásico: estructuras jerárquicas donde una fila referencia
a otra de la misma tabla, como empleados con su jefe, categorías con
subcategorías, o comentarios con respuestas.

Para este ejemplo agregamos una columna `id_categoria_padre` a `categorias`
para simular una jerarquía.

In [28]:
# Agregar columna padre y simular jerarquia
with obtener_conexion() as conn:
    with conn.cursor() as cur:
        cur.execute("""
            ALTER TABLE categorias
            ADD COLUMN IF NOT EXISTS id_padre INTEGER REFERENCES categorias(id);
        """)
        # Electronica y Ropa son subcategorias de Hogar (solo para el ejemplo)
        cur.execute("""
            UPDATE categorias SET id_padre = 3
            WHERE nombre IN ('Electronica', 'Deportes');
        """)

print("--- Categorias con su categoria padre (SELF JOIN) ---")
resultado = ejecutar_query("""
    SELECT
        hijo.nombre     AS categoria,
        padre.nombre    AS categoria_padre
    FROM categorias hijo
    LEFT JOIN categorias padre ON hijo.id_padre = padre.id
    ORDER BY categoria_padre NULLS FIRST, hijo.nombre;
""")
for r in resultado:
    padre = r['categoria_padre'] if r['categoria_padre'] else 'raiz'
    print(f"{r['categoria']:15} → {padre}")

--- Categorias con su categoria padre (SELF JOIN) ---
Hogar           → raiz
Ropa            → raiz
Deportes        → Hogar
Electronica     → Hogar


### CROSS JOIN

Genera el producto cartesiano: combina cada fila de la tabla izquierda
con cada fila de la tabla derecha. Si la tabla A tiene 4 filas y la B tiene 5,
el resultado tiene 4 × 5 = 20 filas.

No tiene condición `ON` — simplemente combina todo con todo.
Se usa para generar combinaciones: tallas con colores, fechas con sucursales,
productos con descuentos posibles.

⚠️ Con tablas grandes el resultado crece exponencialmente. Usar con cuidado.

In [29]:
# Todas las combinaciones de clientes y categorias
print("--- CROSS JOIN clientes x categorias ---")
resultado = ejecutar_query("""
    SELECT
        c.nombre    AS cliente,
        cat.nombre  AS categoria
    FROM clientes c
    CROSS JOIN categorias cat
    WHERE c.email IS NOT NULL
    ORDER BY c.nombre, cat.nombre;
""")
for r in resultado:
    print(f"{r['cliente']:20} x  {r['categoria']}")

print(f"\nTotal combinaciones: {len(resultado)}")

--- CROSS JOIN clientes x categorias ---
Ana Torres           x  Deportes
Ana Torres           x  Electronica
Ana Torres           x  Hogar
Ana Torres           x  Ropa
Carlos Ruiz          x  Deportes
Carlos Ruiz          x  Electronica
Carlos Ruiz          x  Hogar
Carlos Ruiz          x  Ropa
Luis Gomez           x  Deportes
Luis Gomez           x  Electronica
Luis Gomez           x  Hogar
Luis Gomez           x  Ropa
Maria Lopez          x  Deportes
Maria Lopez          x  Electronica
Maria Lopez          x  Hogar
Maria Lopez          x  Ropa

Total combinaciones: 16


## 07. Subconsultas

Una subconsulta es una query anidada dentro de otra query.
El resultado de la subconsulta se usa como valor, conjunto o tabla
dentro de la query exterior.

Se pueden ubicar en tres lugares:

| Ubicación | Nombre | Devuelve |
|-----------|--------|----------|
| En el `WHERE` | Subconsulta de filtro | Un valor o conjunto de valores |
| En el `FROM` | Subconsulta como tabla | Una tabla temporal |
| En el `SELECT` | Subconsulta escalar | Un único valor por fila |

Regla general: si la subconsulta devuelve un solo valor se compara con `=`,
si devuelve múltiples valores se usa `IN`, `ANY` o `ALL`.

### Subconsulta escalar en WHERE

Devuelve exactamente un valor (una fila, una columna) y se usa
en una comparación directa con `=`, `>`, `<`, etc.

Si la subconsulta devuelve más de una fila, PostgreSQL lanza un error.
Cuando no se puede garantizar un único resultado, usar `IN` en su lugar.

In [30]:
# Productos con precio mayor al promedio
print("--- Productos con precio mayor al promedio ---")
resultado = ejecutar_query("""
    SELECT nombre, precio
    FROM productos
    WHERE precio > (SELECT AVG(precio) FROM productos)
    ORDER BY precio DESC;
""")
promedio = ejecutar_query("SELECT ROUND(AVG(precio), 2) AS avg FROM productos;")
print(f"Precio promedio: ${promedio[0]['avg']}")
print()
for r in resultado:
    print(f"{r['nombre']:20} ${r['precio']}")

print()

# El pedido de mayor valor
print("--- Pedido de mayor valor ---")
resultado = ejecutar_query("""
    SELECT id, id_cliente, total, estado
    FROM pedidos
    WHERE total = (SELECT MAX(total) FROM pedidos);
""")
for r in resultado:
    print(f"Pedido {r['id']}  cliente {r['id_cliente']}  ${r['total']}  {r['estado']}")

--- Productos con precio mayor al promedio ---
Precio promedio: $284.99

Laptop 15"           $1199.99

--- Pedido de mayor valor ---
Pedido 1  cliente 1  $1389.98  pagado


### Subconsulta con IN

Cuando la subconsulta devuelve múltiples filas, usamos `IN` para
verificar si el valor pertenece al conjunto resultante.

```sql
WHERE columna IN (SELECT columna FROM otra_tabla WHERE condicion)
```

Es equivalente a un `INNER JOIN` en muchos casos, pero la sintaxis
con `IN` suele ser más legible cuando la intención es filtrar
por pertenencia a un conjunto.

In [31]:
# Clientes que tienen al menos un pedido pagado
print("--- Clientes con pedidos pagados ---")
resultado = ejecutar_query("""
    SELECT nombre, email
    FROM clientes
    WHERE id IN (
        SELECT id_cliente
        FROM pedidos
        WHERE estado = 'pagado'
    )
    ORDER BY nombre;
""")
for r in resultado:
    print(f"{r['nombre']:20} {r['email']}")

print()

# Productos de categorias con mas de 1 producto
print("--- Productos en categorias con mas de 1 producto ---")
resultado = ejecutar_query("""
    SELECT nombre, precio, id_categoria
    FROM productos
    WHERE id_categoria IN (
        SELECT id_categoria
        FROM productos
        GROUP BY id_categoria
        HAVING COUNT(*) > 1
    )
    ORDER BY id_categoria, precio;
""")
for r in resultado:
    print(f"{r['nombre']:20} ${r['precio']}  cat:{r['id_categoria']}")

--- Clientes con pedidos pagados ---


Ana Torres           ana@email.com
Maria Lopez          maria@email.com

--- Productos en categorias con mas de 1 producto ---
Auriculares          $89.99  cat:1
Laptop 15"           $1199.99  cat:1


### Subconsulta con NOT IN

Complemento de `IN`: devuelve las filas cuyo valor **no** pertenece
al conjunto resultante de la subconsulta.

⚠️ Si el conjunto resultante contiene algún `NULL`, `NOT IN` devuelve
`NULL` para todas las filas y el filtro no funciona como se espera.
En ese caso es más seguro usar `NOT EXISTS`.

In [32]:
# Clientes que nunca hicieron un pedido
print("--- Clientes sin pedidos ---")
resultado = ejecutar_query("""
    SELECT nombre, email
    FROM clientes
    WHERE id NOT IN (
        SELECT DISTINCT id_cliente
        FROM pedidos
        WHERE id_cliente IS NOT NULL
    )
    ORDER BY nombre;
""")
for r in resultado:
    email = r['email'] if r['email'] else 'sin email'
    print(f"{r['nombre']:20} {email}")

print()

# Categorias sin productos asignados
print("--- Categorias sin productos ---")
resultado = ejecutar_query("""
    SELECT nombre
    FROM categorias
    WHERE id NOT IN (
        SELECT DISTINCT id_categoria
        FROM productos
        WHERE id_categoria IS NOT NULL
    );
""")
for r in resultado:
    print(r['nombre'])

--- Clientes sin pedidos ---
Carlos Ruiz          carlos@email.com
Sin Email            sin email

--- Categorias sin productos ---


### Subconsulta correlacionada

Una subconsulta correlacionada referencia columnas de la query exterior.
Se ejecuta una vez por cada fila de la query exterior, lo que la hace
más expresiva pero también más costosa en rendimiento.

```sql
SELECT *
FROM tabla_exterior e
WHERE condicion = (
    SELECT funcion(columna)
    FROM tabla_interior i
    WHERE i.columna = e.columna  -- referencia a la query exterior
);
```

Cuando el rendimiento importa, suele reemplazarse por un JOIN o una CTE.
Para volúmenes pequeños o medianos es perfectamente válida.

In [33]:
# Para cada cliente, su pedido de mayor valor
print("--- Pedido mas caro por cliente ---")
resultado = ejecutar_query("""
    SELECT
        c.nombre    AS cliente,
        p.id        AS pedido_id,
        p.total,
        p.estado
    FROM pedidos p
    INNER JOIN clientes c ON p.id_cliente = c.id
    WHERE p.total = (
        SELECT MAX(p2.total)
        FROM pedidos p2
        WHERE p2.id_cliente = p.id_cliente
    )
    ORDER BY p.total DESC;
""")
for r in resultado:
    print(f"{r['cliente']:20} Pedido {r['pedido_id']}  ${r['total']}  {r['estado']}")

--- Pedido mas caro por cliente ---
Ana Torres           Pedido 1  $1389.98  pagado
Maria Lopez          Pedido 4  $89.99  pagado
Luis Gomez           Pedido 2  $79.99  enviado


### EXISTS y NOT EXISTS

`EXISTS` verifica si la subconsulta devuelve al menos una fila.
No le importa el valor devuelto — solo si existe o no existe resultado.

Ventajas sobre `IN`:
- Más eficiente cuando la subconsulta devuelve muchas filas, porque
  PostgreSQL puede detenerse al encontrar la primera coincidencia.
- Seguro con `NULL` — `NOT EXISTS` funciona correctamente aunque
  haya nulos en los datos, a diferencia de `NOT IN`.

```sql
WHERE EXISTS (SELECT 1 FROM tabla WHERE condicion)
```

El `SELECT 1` es convención — el valor no importa, solo la existencia de filas.

In [34]:
# Clientes que tienen al menos un pedido enviado
print("--- Clientes con pedidos enviados (EXISTS) ---")
resultado = ejecutar_query("""
    SELECT nombre, email
    FROM clientes c
    WHERE EXISTS (
        SELECT 1
        FROM pedidos p
        WHERE p.id_cliente = c.id
        AND p.estado = 'enviado'
    )
    ORDER BY nombre;
""")
for r in resultado:
    print(f"{r['nombre']:20} {r['email']}")

print()

# Clientes sin ningun pedido (NOT EXISTS)
print("--- Clientes sin pedidos (NOT EXISTS) ---")
resultado = ejecutar_query("""
    SELECT nombre, email
    FROM clientes c
    WHERE NOT EXISTS (
        SELECT 1
        FROM pedidos p
        WHERE p.id_cliente = c.id
    )
    ORDER BY nombre;
""")
for r in resultado:
    email = r['email'] if r['email'] else 'sin email'
    print(f"{r['nombre']:20} {email}")

--- Clientes con pedidos enviados (EXISTS) ---
Ana Torres           ana@email.com
Luis Gomez           luis@email.com

--- Clientes sin pedidos (NOT EXISTS) ---
Carlos Ruiz          carlos@email.com
Sin Email            sin email


### Subconsulta en FROM — tabla derivada

Una subconsulta en el `FROM` actúa como una tabla temporal.
Debe llevar un alias obligatoriamente.

Útil cuando necesitamos operar sobre un resultado agregado —
por ejemplo, filtrar sobre un `COUNT` o calcular el promedio
de un promedio. Sin tabla derivada esto no sería posible
en una sola query.

In [35]:
# Promedio del total de pedidos por cliente,
# luego filtrar los que estan por encima del promedio general
print("--- Clientes con gasto promedio sobre la media general ---")
resultado = ejecutar_query("""
    SELECT
        cliente,
        ROUND(promedio_cliente, 2) AS promedio_cliente,
        ROUND(media_general, 2)    AS media_general
    FROM (
        SELECT
            c.nombre            AS cliente,
            AVG(p.total)        AS promedio_cliente,
            AVG(AVG(p.total)) OVER () AS media_general
        FROM pedidos p
        INNER JOIN clientes c ON p.id_cliente = c.id
        GROUP BY c.id, c.nombre
    ) AS resumen
    WHERE promedio_cliente > media_general
    ORDER BY promedio_cliente DESC;
""")
for r in resultado:
    print(f"{r['cliente']:20} avg: ${r['promedio_cliente']}  media: ${r['media_general']}")

--- Clientes con gasto promedio sobre la media general ---
Ana Torres           avg: $712.49  media: $294.16


## 08. Normalización — 1NF, 2NF, 3NF

La normalización es el proceso de organizar las tablas de una base de datos
para eliminar redundancia y evitar anomalías al insertar, actualizar o eliminar datos.

Una **anomalía** es un problema que surge cuando los datos están mal organizados:
- **Anomalía de inserción** — no se puede agregar un dato sin agregar otro innecesario
- **Anomalía de actualización** — cambiar un dato requiere modificarlo en múltiples filas
- **Anomalía de eliminación** — eliminar una fila hace perder información que no debería perderse

La normalización se organiza en **formas normales** (NF), cada una más estricta que la anterior.
Cumplir la 3NF es suficiente para la gran mayoría de sistemas reales.

| Forma Normal | Requisito principal |
|--------------|---------------------|
| 1NF | Valores atómicos, sin grupos repetidos |
| 2NF | Cumple 1NF + sin dependencias parciales |
| 3NF | Cumple 2NF + sin dependencias transitivas |

### Primera Forma Normal — 1NF

Una tabla está en 1NF si:
1. Cada celda contiene un único valor atómico (no listas, no arrays de texto)
2. Cada columna tiene un tipo de dato consistente
3. Cada fila es única (tiene una Primary Key)

#### Ejemplo de violación

| id_pedido | cliente | productos |
|-----------|---------|-----------|
| 1 | Ana | Laptop, Auriculares |
| 2 | Luis | Zapatillas |

La columna `productos` viola 1NF: contiene múltiples valores en una celda.
Si necesitamos buscar todos los pedidos que incluyen "Laptop", no podemos
hacerlo con SQL estándar — tendríamos que parsear el string.

#### Forma correcta

Separar en una tabla de detalle con una fila por producto por pedido.
Esto además permite agregar cantidad, precio unitario, descuento, etc.

In [36]:
# Tabla que viola 1NF (solo para ilustrar — no la vamos a usar)
with obtener_conexion() as conn:
    with conn.cursor() as cur:
        cur.execute("""
            CREATE TABLE IF NOT EXISTS pedidos_mal_disenado (
                id_pedido   INTEGER PRIMARY KEY,
                cliente     VARCHAR(100),
                productos   TEXT  -- viola 1NF: "Laptop, Auriculares"
            );
        """)
        cur.execute("""
            INSERT INTO pedidos_mal_disenado VALUES
            (1, 'Ana',  'Laptop, Auriculares'),
            (2, 'Luis', 'Zapatillas')
            ON CONFLICT DO NOTHING;
        """)

print("--- Tabla que viola 1NF ---")
resultado = ejecutar_query("SELECT * FROM pedidos_mal_disenado;")
for r in resultado:
    print(f"Pedido {r['id_pedido']}  {r['cliente']:10} productos: {r['productos']}")

print()

# Forma correcta: tabla de detalle
with obtener_conexion() as conn:
    with conn.cursor() as cur:
        cur.execute("""
            CREATE TABLE IF NOT EXISTS pedido_detalle (
                id              SERIAL PRIMARY KEY,
                id_pedido       INTEGER REFERENCES pedidos(id),
                id_producto     INTEGER REFERENCES productos(id),
                cantidad        INTEGER NOT NULL CHECK (cantidad > 0),
                precio_unitario NUMERIC(10, 2) NOT NULL
            );
        """)
        cur.execute("""
            INSERT INTO pedido_detalle (id_pedido, id_producto, cantidad, precio_unitario)
            VALUES
                (1, 1, 1, 1199.99),
                (1, 2, 1,   89.99),
                (2, 4, 1,   79.99),
                (3, 5, 1,   34.99),
                (4, 2, 1,   89.99)
            ON CONFLICT DO NOTHING;
        """)

print("--- Tabla en 1NF (pedido_detalle) ---")
resultado = ejecutar_query("""
    SELECT
        pd.id_pedido,
        p.nombre    AS producto,
        pd.cantidad,
        pd.precio_unitario
    FROM pedido_detalle pd
    INNER JOIN productos p ON pd.id_producto = p.id
    ORDER BY pd.id_pedido;
""")
for r in resultado:
    print(f"Pedido {r['id_pedido']}  {r['producto']:20} x{r['cantidad']}  ${r['precio_unitario']}")

--- Tabla que viola 1NF ---
Pedido 1  Ana        productos: Laptop, Auriculares
Pedido 2  Luis       productos: Zapatillas

--- Tabla en 1NF (pedido_detalle) ---
Pedido 1  Auriculares          x1  $89.99
Pedido 1  Laptop 15"           x1  $1199.99
Pedido 1  Auriculares          x1  $89.99
Pedido 1  Laptop 15"           x1  $1199.99
Pedido 1  Auriculares          x1  $89.99
Pedido 1  Laptop 15"           x1  $1199.99
Pedido 1  Laptop 15"           x1  $1199.99
Pedido 1  Auriculares          x1  $89.99
Pedido 1  Laptop 15"           x1  $1199.99
Pedido 1  Auriculares          x1  $89.99
Pedido 1  Laptop 15"           x1  $1199.99
Pedido 1  Auriculares          x1  $89.99
Pedido 1  Laptop 15"           x1  $1199.99
Pedido 1  Auriculares          x1  $89.99
Pedido 1  Laptop 15"           x1  $1199.99
Pedido 1  Auriculares          x1  $89.99
Pedido 1  Laptop 15"           x1  $1199.99
Pedido 1  Auriculares          x1  $89.99
Pedido 1  Laptop 15"           x1  $1199.99
Pedido 1  Auriculare

### Segunda Forma Normal — 2NF

Una tabla está en 2NF si:
1. Está en 1NF
2. Todos los atributos no clave dependen de **toda** la clave primaria,
   no solo de una parte de ella

Esto solo aplica cuando la Primary Key es compuesta (más de una columna).
Si la PK es simple (una sola columna), la tabla automáticamente cumple 2NF.

#### Ejemplo de violación

Supongamos una tabla con PK compuesta `(id_pedido, id_producto)`:

| id_pedido | id_producto | cantidad | nombre_producto | categoria |
|-----------|-------------|----------|-----------------|-----------|
| 1 | 101 | 2 | Laptop | Electronica |
| 1 | 102 | 1 | Auriculares | Electronica |

`nombre_producto` y `categoria` dependen solo de `id_producto`,
no de la combinación `(id_pedido, id_producto)`.
Eso es una dependencia parcial — viola 2NF.

Si cambia el nombre de un producto hay que actualizarlo en todas las filas
donde aparece ese producto. Eso es una anomalía de actualización.

#### Forma correcta

Mover `nombre_producto` y `categoria` a su propia tabla `productos`,
y en `pedido_detalle` solo guardar `id_producto` como referencia.
Así es exactamente como diseñamos nuestro schema.

In [37]:
# Verificar que pedido_detalle esta en 2NF
# La PK es 'id' (simple), y cada columna depende de ella completamente
print("--- Estructura de pedido_detalle (en 2NF) ---")
resultado = ejecutar_query("""
    SELECT column_name, data_type, is_nullable
    FROM information_schema.columns
    WHERE table_name = 'pedido_detalle'
    ORDER BY ordinal_position;
""")
for r in resultado:
    nullable = 'NULL' if r['is_nullable'] == 'YES' else 'NOT NULL'
    print(f"{r['column_name']:20} {r['data_type']:20} {nullable}")

print()

# Comparar con la tabla mal disenada que mezcla datos de producto en el detalle
print("--- Tabla que violaría 2NF ---")
with obtener_conexion() as conn:
    with conn.cursor() as cur:
        cur.execute("""
            CREATE TABLE IF NOT EXISTS detalle_mal_disenado (
                id_pedido        INTEGER,
                id_producto      INTEGER,
                cantidad         INTEGER,
                nombre_producto  VARCHAR(200),  -- depende solo de id_producto
                categoria        VARCHAR(100),  -- depende solo de id_producto
                PRIMARY KEY (id_pedido, id_producto)
            );
        """)

resultado = ejecutar_query("""
    SELECT column_name FROM information_schema.columns
    WHERE table_name = 'detalle_mal_disenado'
    ORDER BY ordinal_position;
""")
print("Columnas:", [r['column_name'] for r in resultado])
print("nombre_producto y categoria dependen solo de id_producto → viola 2NF")

--- Estructura de pedido_detalle (en 2NF) ---


id                   integer              NOT NULL
id_pedido            integer              NULL
id_producto          integer              NULL
cantidad             integer              NOT NULL
precio_unitario      numeric              NOT NULL

--- Tabla que violaría 2NF ---
Columnas: ['id_pedido', 'id_producto', 'cantidad', 'nombre_producto', 'categoria']
nombre_producto y categoria dependen solo de id_producto → viola 2NF


### Tercera Forma Normal — 3NF

Una tabla está en 3NF si:
1. Está en 2NF
2. No existen **dependencias transitivas**: ningún atributo no clave
   depende de otro atributo no clave

Una dependencia transitiva ocurre cuando:
`PK → columna_A → columna_B`

Es decir, `columna_B` depende de `columna_A` que a su vez depende de la PK,
en lugar de depender directamente de la PK.

#### Ejemplo de violación

| id_pedido | id_cliente | email_cliente | ciudad_cliente |
|-----------|------------|---------------|----------------|
| 1 | 10 | ana@mail.com | Buenos Aires |
| 2 | 11 | luis@mail.com | Cordoba |

`email_cliente` y `ciudad_cliente` dependen de `id_cliente`,
no directamente de `id_pedido`. Si Ana cambia su email,
hay que actualizarlo en todos sus pedidos — anomalía de actualización.

#### Forma correcta

Los datos del cliente van en su propia tabla `clientes`.
En `pedidos` solo se guarda `id_cliente` como FK.
Exactamente como está diseñado nuestro schema.

In [38]:
# Verificar que nuestro schema cumple 3NF
print("--- Schema actual: sin dependencias transitivas ---")
resultado = ejecutar_query("""
    SELECT
        tc.table_name,
        kcu.column_name         AS columna_fk,
        ccu.table_name          AS tabla_referenciada,
        ccu.column_name         AS columna_referenciada
    FROM information_schema.table_constraints tc
    JOIN information_schema.key_column_usage kcu
        ON tc.constraint_name = kcu.constraint_name
    JOIN information_schema.constraint_column_usage ccu
        ON ccu.constraint_name = tc.constraint_name
    WHERE tc.constraint_type = 'FOREIGN KEY'
    ORDER BY tc.table_name;
""")
for r in resultado:
    print(f"{r['table_name']:20} {r['columna_fk']:20} → {r['tabla_referenciada']}.{r['columna_referenciada']}")

print()

# Tabla que violaría 3NF
print("--- Tabla que violaría 3NF ---")
with obtener_conexion() as conn:
    with conn.cursor() as cur:
        cur.execute("""
            CREATE TABLE IF NOT EXISTS pedidos_mal_3nf (
                id_pedido       INTEGER PRIMARY KEY,
                id_cliente      INTEGER,
                email_cliente   VARCHAR(200),  -- depende de id_cliente, no de id_pedido
                ciudad_cliente  VARCHAR(100),  -- depende de id_cliente, no de id_pedido
                total           NUMERIC(10,2)
            );
        """)

print("email_cliente y ciudad_cliente dependen de id_cliente → viola 3NF")
print("Solucion: mover esos datos a la tabla clientes y referenciar por id_cliente")

--- Schema actual: sin dependencias transitivas ---


categorias           id_padre             → categorias.id
pedidos              id_cliente           → clientes.id
productos            id_categoria         → categorias.id

--- Tabla que violaría 3NF ---
email_cliente y ciudad_cliente dependen de id_cliente → viola 3NF
Solucion: mover esos datos a la tabla clientes y referenciar por id_cliente


### Nuestro schema en 3NF

El schema e-commerce que construimos cumple las tres formas normales:

| Tabla | Columnas propias | FK |
|-------|-----------------|-----|
| `categorias` | id, nombre, activa | — |
| `productos` | id, nombre, precio, stock, creado_en | id_categoria → categorias |
| `clientes` | id, nombre, email, creado_en | — |
| `pedidos` | id, total, estado, creado_en | id_cliente → clientes |
| `pedido_detalle` | id, cantidad, precio_unitario | id_pedido → pedidos, id_producto → productos |

- **1NF** — cada celda tiene un valor atómico, `pedido_detalle` separa los productos por fila
- **2NF** — todas las columnas dependen de la PK completa de su tabla
- **3NF** — no hay dependencias transitivas: los datos de cada entidad viven en su propia tabla

In [39]:
# Vista completa del schema normalizado
print("--- Schema completo normalizado ---")
resultado = ejecutar_query("""
    SELECT
        t.table_name,
        COUNT(c.column_name) AS columnas
    FROM information_schema.tables t
    JOIN information_schema.columns c
        ON c.table_name = t.table_name
        AND c.table_schema = 'public'
    WHERE t.table_schema = 'public'
    AND t.table_type = 'BASE TABLE'
    AND t.table_name NOT LIKE '%mal%'
    GROUP BY t.table_name
    ORDER BY t.table_name;
""")
for r in resultado:
    print(f"{r['table_name']:25} {r['columnas']} columnas")

# Limpiar tablas de ejemplo creadas para ilustrar violaciones
with obtener_conexion() as conn:
    with conn.cursor() as cur:
        cur.execute("""
            DROP TABLE IF EXISTS pedidos_mal_disenado;
            DROP TABLE IF EXISTS detalle_mal_disenado;
            DROP TABLE IF EXISTS pedidos_mal_3nf;
        """)
print("\nTablas de ejemplo eliminadas")

--- Schema completo normalizado ---
categorias                4 columnas
clientes                  4 columnas
pedido_detalle            5 columnas
pedidos                   5 columnas
productos                 6 columnas

Tablas de ejemplo eliminadas


## 09. Índices y EXPLAIN ANALYZE

Un índice es una estructura de datos auxiliar que PostgreSQL mantiene
junto a la tabla para acelerar las búsquedas. Funciona igual que el
índice de un libro: en lugar de leer página por página, vas directo
al número de página.

Sin índice → PostgreSQL lee todas las filas de la tabla (**Sequential Scan**).
Con índice → PostgreSQL va directo a las filas relevantes (**Index Scan**).

El costo de un índice:
- Ocupa espacio en disco
- Cada `INSERT`, `UPDATE` y `DELETE` debe actualizar también el índice
- Por eso no se indexa todo — solo las columnas que se consultan frecuentemente

PostgreSQL crea índices automáticamente para `PRIMARY KEY` y `UNIQUE`.
El resto hay que crearlos manualmente según los patrones de consulta.

| Tipo | Cuándo usarlo |
|------|---------------|
| `BTREE` | Default. Comparaciones `=`, `<`, `>`, `BETWEEN`, `ORDER BY` |
| `HASH` | Solo igualdad exacta `=`. Raramente mejor que BTREE |
| `GIN` | Arrays, búsqueda full-text, JSON |
| `GIST` | Datos geométricos, rangos |

### EXPLAIN ANALYZE

`EXPLAIN` muestra el plan de ejecución que PostgreSQL decide usar
para una query, sin ejecutarla.

`EXPLAIN ANALYZE` lo ejecuta y además muestra los tiempos reales.

Los términos clave del plan:

| Término | Significado |
|---------|-------------|
| `Seq Scan` | Lectura secuencial de toda la tabla — sin índice |
| `Index Scan` | Usa el índice para ir directo a las filas |
| `Bitmap Index Scan` | Usa el índice para construir un mapa de filas, luego las lee |
| `cost` | Estimación del costo: `inicio..total`. Unidades arbitrarias, sirve para comparar |
| `rows` | Filas estimadas que devolverá el nodo |
| `actual time` | Tiempo real en milisegundos |
| `loops` | Cuántas veces se ejecutó ese nodo |

El objetivo es evitar `Seq Scan` en tablas grandes con filtros selectivos.

In [40]:
# Ver el plan de una query sin indice
print("--- EXPLAIN sin indice (busqueda por email) ---")
resultado = ejecutar_query("""
    EXPLAIN ANALYZE
    SELECT * FROM clientes
    WHERE email = 'ana@email.com';
""")
for r in resultado:
    print(r['QUERY PLAN'])

--- EXPLAIN sin indice (busqueda por email) ---
Index Scan using clientes_email_key on clientes  (cost=0.14..8.16 rows=1 width=744) (actual time=0.042..0.042 rows=1 loops=1)
  Index Cond: ((email)::text = 'ana@email.com'::text)
Planning Time: 0.335 ms
Execution Time: 0.153 ms


### Crear un índice

```sql
CREATE INDEX nombre_indice ON tabla (columna);
```

Convención de nombres: `idx_tabla_columna`.
Así queda claro sobre qué tabla y columna actúa sin tener que inspeccionarlo.

`CREATE INDEX CONCURRENTLY` crea el índice sin bloquear la tabla —
importante en producción donde hay escrituras mientras se indexa.
En desarrollo y notebooks usamos el `CREATE INDEX` normal.

In [41]:
# Crear indices utiles para nuestro schema
with obtener_conexion() as conn:
    with conn.cursor() as cur:

        # Email de clientes — busquedas frecuentes por email
        cur.execute("""
            CREATE INDEX IF NOT EXISTS idx_clientes_email
            ON clientes (email);
        """)

        # FK de productos — JOINs con categorias
        cur.execute("""
            CREATE INDEX IF NOT EXISTS idx_productos_id_categoria
            ON productos (id_categoria);
        """)

        # FK y estado de pedidos — filtros frecuentes
        cur.execute("""
            CREATE INDEX IF NOT EXISTS idx_pedidos_id_cliente
            ON pedidos (id_cliente);
        """)
        cur.execute("""
            CREATE INDEX IF NOT EXISTS idx_pedidos_estado
            ON pedidos (estado);
        """)

        # FK de pedido_detalle — JOINs frecuentes
        cur.execute("""
            CREATE INDEX IF NOT EXISTS idx_detalle_id_pedido
            ON pedido_detalle (id_pedido);
        """)
        cur.execute("""
            CREATE INDEX IF NOT EXISTS idx_detalle_id_producto
            ON pedido_detalle (id_producto);
        """)

print("Indices creados")

Indices creados


### Ver los índices existentes

`pg_indexes` es una vista del sistema que lista todos los índices
con su tabla, nombre y la sentencia DDL que los creó.

PostgreSQL incluye automáticamente los índices de `PRIMARY KEY` y `UNIQUE`
— se pueden ver aquí junto a los que creamos manualmente.

In [42]:
print("--- Indices en el schema ---")
resultado = ejecutar_query("""
    SELECT
        tablename   AS tabla,
        indexname   AS indice,
        indexdef    AS definicion
    FROM pg_indexes
    WHERE schemaname = 'public'
    ORDER BY tablename, indexname;
""")
for r in resultado:
    print(f"{r['tabla']:20} {r['indice']:35} {r['definicion']}")

--- Indices en el schema ---
categorias           categorias_pkey                     CREATE UNIQUE INDEX categorias_pkey ON public.categorias USING btree (id)
clientes             clientes_email_key                  CREATE UNIQUE INDEX clientes_email_key ON public.clientes USING btree (email)
clientes             clientes_pkey                       CREATE UNIQUE INDEX clientes_pkey ON public.clientes USING btree (id)
clientes             idx_clientes_email                  CREATE INDEX idx_clientes_email ON public.clientes USING btree (email)
pedido_detalle       idx_detalle_id_pedido               CREATE INDEX idx_detalle_id_pedido ON public.pedido_detalle USING btree (id_pedido)
pedido_detalle       idx_detalle_id_producto             CREATE INDEX idx_detalle_id_producto ON public.pedido_detalle USING btree (id_producto)
pedido_detalle       pedido_detalle_pkey                 CREATE UNIQUE INDEX pedido_detalle_pkey ON public.pedido_detalle USING btree (id)
pedidos              idx_

### Comparar plan antes y después del índice

Con pocos registros PostgreSQL puede ignorar el índice y hacer Seq Scan
de todas formas — el planner decide que es más barato leer toda la tabla
que usar el índice cuando la tabla es pequeña.

Para forzar la comparación desactivamos el Seq Scan temporalmente.
Esto es solo para aprendizaje — nunca se hace en producción.

En tablas con miles o millones de filas el índice aparece solo,
sin necesidad de forzarlo.

In [43]:
# Con indice activo (comportamiento normal)
print("--- EXPLAIN con indice (email) ---")
resultado = ejecutar_query("""
    EXPLAIN ANALYZE
    SELECT * FROM clientes
    WHERE email = 'ana@email.com';
""")
for r in resultado:
    print(r['QUERY PLAN'])

print()

# Forzar seq scan desactivando index scan para comparar
print("--- EXPLAIN forzando Seq Scan (para comparar) ---")
with obtener_conexion() as conn:
    with conn.cursor() as cur:
        cur.execute("SET enable_indexscan = OFF;")
        cur.execute("SET enable_bitmapscan = OFF;")
        cur.execute("""
            EXPLAIN ANALYZE
            SELECT * FROM clientes
            WHERE email = 'ana@email.com';
        """)
        resultado = cur.fetchall()
        for r in resultado:
            print(r['QUERY PLAN'])

--- EXPLAIN con indice (email) ---


Seq Scan on clientes  (cost=0.00..1.06 rows=1 width=744) (actual time=0.032..0.034 rows=1 loops=1)
  Filter: ((email)::text = 'ana@email.com'::text)
  Rows Removed by Filter: 4
Planning Time: 1.679 ms
Execution Time: 0.108 ms

--- EXPLAIN forzando Seq Scan (para comparar) ---
Seq Scan on clientes  (cost=0.00..1.06 rows=1 width=744) (actual time=0.040..0.042 rows=1 loops=1)
  Filter: ((email)::text = 'ana@email.com'::text)
  Rows Removed by Filter: 4
Planning Time: 0.704 ms
Execution Time: 0.101 ms


### Índice compuesto

Un índice puede cubrir más de una columna. Es útil cuando las queries
filtran frecuentemente por la combinación de dos columnas juntas.

```sql
CREATE INDEX idx_tabla_col1_col2 ON tabla (col1, col2);
```

El orden de las columnas importa: el índice es útil para queries que
filtran por `col1` sola o por `col1 + col2` juntas.
No es útil para queries que filtran solo por `col2`.

Regla general: la columna más selectiva (la que filtra más filas) va primero.

In [44]:
# Indice compuesto para busquedas por cliente + estado en pedidos
with obtener_conexion() as conn:
    with conn.cursor() as cur:
        cur.execute("""
            CREATE INDEX IF NOT EXISTS idx_pedidos_cliente_estado
            ON pedidos (id_cliente, estado);
        """)

print("--- EXPLAIN con indice compuesto ---")
resultado = ejecutar_query("""
    EXPLAIN ANALYZE
    SELECT id, total, estado
    FROM pedidos
    WHERE id_cliente = 1
    AND estado = 'pagado';
""")
for r in resultado:
    print(r['QUERY PLAN'])

--- EXPLAIN con indice compuesto ---
Seq Scan on pedidos  (cost=0.00..1.06 rows=1 width=138) (actual time=0.015..0.016 rows=1 loops=1)
  Filter: ((id_cliente = 1) AND ((estado)::text = 'pagado'::text))
  Rows Removed by Filter: 3
Planning Time: 0.540 ms
Execution Time: 0.051 ms


### Cuándo eliminar un índice

Un índice que nadie usa es puro costo: ocupa disco y ralentiza escrituras.
PostgreSQL lleva estadísticas de uso en `pg_stat_user_indexes`.

Si `idx_scan = 0` después de un período de uso normal,
el índice probablemente no sirve y conviene eliminarlo.

```sql
DROP INDEX nombre_indice;
```

In [45]:
# Ver estadisticas de uso de indices
print("--- Uso de indices ---")
resultado = ejecutar_query("""
    SELECT
        i.indexrelname      AS indice,
        t.relname           AS tabla,
        i.idx_scan          AS veces_usado,
        i.idx_tup_read      AS filas_leidas,
        i.idx_tup_fetch     AS filas_obtenidas
    FROM pg_stat_user_indexes i
    JOIN pg_stat_user_tables t ON i.relid = t.relid
    ORDER BY i.idx_scan DESC;
""")
for r in resultado:
    print(f"{r['indice']:40} {r['tabla']:20} scans: {r['veces_usado']}")

print()

# Eliminar un indice de ejemplo
with obtener_conexion() as conn:
    with conn.cursor() as cur:
        cur.execute("DROP INDEX IF EXISTS idx_pedidos_estado;")

print("idx_pedidos_estado eliminado")
print("(lo eliminamos para ilustrar DROP INDEX — en la practica se evaluaria su uso primero)")

--- Uso de indices ---
pedido_detalle_pkey                      pedido_detalle       scans: 55
categorias_pkey                          categorias           scans: 7
clientes_pkey                            clientes             scans: 4
clientes_email_key                       clientes             scans: 3
pedidos_pkey                             pedidos              scans: 1
idx_clientes_email                       clientes             scans: 0
idx_detalle_id_producto                  pedido_detalle       scans: 0
idx_pedidos_id_cliente                   pedidos              scans: 0
idx_pedidos_estado                       pedidos              scans: 0
idx_pedidos_cliente_estado               pedidos              scans: 0
idx_detalle_id_pedido                    pedido_detalle       scans: 0
productos_pkey                           productos            scans: 0
idx_productos_id_categoria               productos            scans: 0

idx_pedidos_estado eliminado
(lo eliminamos para ilu

## 10. Window Functions — ROW_NUMBER, RANK, DENSE_RANK, LAG, LEAD, PARTITION BY

Una window function calcula un valor para cada fila basándose en un conjunto
de filas relacionadas — la "ventana" — sin colapsar las filas en un grupo
como hace `GROUP BY`.

Con `GROUP BY`: N filas → 1 fila por grupo
Con window function: N filas → N filas (cada una con el valor calculado)

```sql
FUNCION() OVER (
    PARTITION BY columna   -- divide en grupos (opcional)
    ORDER BY columna       -- orden dentro de cada grupo
)
```

| Función | Descripción |
|---------|-------------|
| `ROW_NUMBER()` | Número de fila único dentro de la ventana, sin empates |
| `RANK()` | Ranking con saltos — si dos filas empatan en 1°, la siguiente es 3° |
| `DENSE_RANK()` | Ranking sin saltos — si dos filas empatan en 1°, la siguiente es 2° |
| `LAG(col, n)` | Valor de la columna n filas hacia atrás |
| `LEAD(col, n)` | Valor de la columna n filas hacia adelante |
| `SUM() OVER` | Suma acumulada o por partición |
| `AVG() OVER` | Promedio por partición sin colapsar filas |

### ROW_NUMBER

Asigna un número secuencial único a cada fila dentro de la ventana.
No hay empates — si dos filas tienen el mismo valor de ordenamiento,
el número se asigna arbitrariamente entre ellas.

Caso de uso típico: obtener el N-ésimo registro de cada grupo,
como el producto más caro de cada categoría.

In [46]:
# Numerar productos por precio dentro de cada categoria
print("--- ROW_NUMBER por categoria ---")
resultado = ejecutar_query("""
    SELECT
        p.nombre        AS producto,
        c.nombre        AS categoria,
        p.precio,
        ROW_NUMBER() OVER (
            PARTITION BY p.id_categoria
            ORDER BY p.precio DESC
        ) AS nro_en_categoria
    FROM productos p
    JOIN categorias c ON p.id_categoria = c.id
    ORDER BY c.nombre, nro_en_categoria;
""")
for r in resultado:
    print(f"{r['categoria']:15} #{r['nro_en_categoria']}  {r['producto']:20} ${r['precio']}")

print()

# Usar ROW_NUMBER para obtener el producto mas caro de cada categoria
print("--- Producto mas caro por categoria ---")
resultado = ejecutar_query("""
    SELECT categoria, producto, precio
    FROM (
        SELECT
            p.nombre        AS producto,
            c.nombre        AS categoria,
            p.precio,
            ROW_NUMBER() OVER (
                PARTITION BY p.id_categoria
                ORDER BY p.precio DESC
            ) AS rn
        FROM productos p
        JOIN categorias c ON p.id_categoria = c.id
    ) sub
    WHERE rn = 1
    ORDER BY precio DESC;
""")
for r in resultado:
    print(f"{r['categoria']:15} {r['producto']:20} ${r['precio']}")

--- ROW_NUMBER por categoria ---
Deportes        #1  Zapatillas 42        $79.99
Electronica     #1  Laptop 15"           $1199.99
Electronica     #2  Auriculares          $89.99
Hogar           #1  Lampara LED          $34.99
Ropa            #1  Camiseta M           $19.99

--- Producto mas caro por categoria ---
Electronica     Laptop 15"           $1199.99
Deportes        Zapatillas 42        $79.99
Hogar           Lampara LED          $34.99
Ropa            Camiseta M           $19.99


### RANK vs DENSE_RANK

Ambas asignan un ranking pero difieren en cómo manejan los empates:

| Posición | RANK | DENSE_RANK |
|----------|------|------------|
| 1° (empate) | 1 | 1 |
| 1° (empate) | 1 | 1 |
| Siguiente | 3 | 2 |

`RANK` — salta números después de un empate (como en una competencia real:
si dos personas llegan primeras, la siguiente es tercera).

`DENSE_RANK` — no salta números (la siguiente posición después de un empate
siempre es consecutiva).

Cuál usar depende del contexto: para rankings de competencia `RANK`,
para categorización por tramos `DENSE_RANK`.

In [47]:
# Insertar productos con precio repetido para ver el efecto del empate
with obtener_conexion() as conn:
    with conn.cursor() as cur:
        cur.execute("""
            INSERT INTO productos (nombre, precio, stock, id_categoria)
            VALUES ('Auriculares Pro', 89.99, 20, 1)
            ON CONFLICT DO NOTHING;
        """)

print("--- RANK vs DENSE_RANK ---")
resultado = ejecutar_query("""
    SELECT
        nombre,
        precio,
        RANK()       OVER (ORDER BY precio DESC) AS rank,
        DENSE_RANK() OVER (ORDER BY precio DESC) AS dense_rank
    FROM productos
    ORDER BY precio DESC;
""")
for r in resultado:
    print(f"{r['nombre']:22} ${r['precio']:8}  RANK:{r['rank']}  DENSE_RANK:{r['dense_rank']}")

--- RANK vs DENSE_RANK ---
Laptop 15"             $ 1199.99  RANK:1  DENSE_RANK:1
Auriculares            $   89.99  RANK:2  DENSE_RANK:2
Auriculares Pro        $   89.99  RANK:2  DENSE_RANK:2
Zapatillas 42          $   79.99  RANK:4  DENSE_RANK:3
Lampara LED            $   34.99  RANK:5  DENSE_RANK:4
Camiseta M             $   19.99  RANK:6  DENSE_RANK:5


### Agregados como window functions

`SUM()`, `AVG()`, `COUNT()` y el resto de los agregados también funcionan
como window functions cuando se usan con `OVER`.

La diferencia clave con `GROUP BY`:
- `GROUP BY` colapsa las filas → un resultado por grupo
- `OVER` mantiene todas las filas → cada fila muestra el agregado de su ventana

Esto permite comparar cada fila con el total o promedio de su grupo
sin perder el detalle fila por fila.

In [48]:
# Precio de cada producto vs promedio de su categoria
print("--- Precio vs promedio de categoria ---")
resultado = ejecutar_query("""
    SELECT
        p.nombre                            AS producto,
        c.nombre                            AS categoria,
        p.precio,
        ROUND(AVG(p.precio) OVER (
            PARTITION BY p.id_categoria
        ), 2)                               AS avg_categoria,
        ROUND(p.precio - AVG(p.precio) OVER (
            PARTITION BY p.id_categoria
        ), 2)                               AS diferencia
    FROM productos p
    JOIN categorias c ON p.id_categoria = c.id
    ORDER BY c.nombre, p.precio DESC;
""")
for r in resultado:
    signo = "+" if r['diferencia'] >= 0 else ""
    print(f"{r['categoria']:15} {r['producto']:22} ${r['precio']:8}  "
          f"avg:${r['avg_categoria']}  dif:{signo}{r['diferencia']}")

print()

# Suma acumulada de pedidos ordenados por fecha
print("--- Suma acumulada de pedidos ---")
resultado = ejecutar_query("""
    SELECT
        id,
        id_cliente,
        total,
        creado_en::DATE             AS fecha,
        SUM(total) OVER (
            ORDER BY creado_en
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        )                           AS total_acumulado
    FROM pedidos
    ORDER BY creado_en;
""")
for r in resultado:
    print(f"Pedido {r['id']}  ${r['total']:8}  acumulado: ${r['total_acumulado']}")

--- Precio vs promedio de categoria ---
Deportes        Zapatillas 42          $   79.99  avg:$79.99  dif:+0.00
Electronica     Laptop 15"             $ 1199.99  avg:$459.99  dif:+740.00
Electronica     Auriculares Pro        $   89.99  avg:$459.99  dif:-370.00
Electronica     Auriculares            $   89.99  avg:$459.99  dif:-370.00
Hogar           Lampara LED            $   34.99  avg:$34.99  dif:+0.00
Ropa            Camiseta M             $   19.99  avg:$19.99  dif:+0.00

--- Suma acumulada de pedidos ---
Pedido 1  $ 1389.98  acumulado: $1389.98
Pedido 2  $   79.99  acumulado: $1469.97
Pedido 4  $   89.99  acumulado: $1559.96
Pedido 3  $   34.99  acumulado: $1594.95


### LAG y LEAD — acceder a filas adyacentes

Permiten acceder al valor de una fila anterior (`LAG`) o posterior (`LEAD`)
dentro de la ventana, sin necesidad de un self JOIN.

```sql
LAG(columna, n, valor_default)  OVER (ORDER BY columna)
LEAD(columna, n, valor_default) OVER (ORDER BY columna)
```

- `n` — cuántas filas hacia atrás o adelante (default: 1)
- `valor_default` — qué devolver si no hay fila anterior/posterior (default: NULL)

Casos de uso típicos:
- Comparar ventas de un mes con el mes anterior
- Calcular la variación entre períodos
- Detectar brechas o saltos en una secuencia

In [49]:
# Comparar total de cada pedido con el anterior del mismo cliente
print("--- Variacion entre pedidos por cliente ---")
resultado = ejecutar_query("""
    SELECT
        id,
        id_cliente,
        total,
        creado_en::DATE                         AS fecha,
        LAG(total) OVER (
            PARTITION BY id_cliente
            ORDER BY creado_en
        )                                       AS pedido_anterior,
        ROUND(total - LAG(total) OVER (
            PARTITION BY id_cliente
            ORDER BY creado_en
        ), 2)                                   AS variacion
    FROM pedidos
    ORDER BY id_cliente, creado_en;
""")
for r in resultado:
    anterior = f"${r['pedido_anterior']}" if r['pedido_anterior'] else "primero"
    variacion = f"${r['variacion']}" if r['variacion'] is not None else "-"
    print(f"Cliente {r['id_cliente']}  Pedido {r['id']}  "
          f"${r['total']}  anterior:{anterior:10}  variacion:{variacion}")

print()

# LEAD: ver el proximo pedido de cada cliente
print("--- Proximo pedido por cliente ---")
resultado = ejecutar_query("""
    SELECT
        id,
        id_cliente,
        total,
        estado,
        LEAD(estado) OVER (
            PARTITION BY id_cliente
            ORDER BY creado_en
        )                   AS proximo_estado,
        LEAD(total) OVER (
            PARTITION BY id_cliente
            ORDER BY creado_en
        )                   AS proximo_total
    FROM pedidos
    ORDER BY id_cliente, id;
""")
for r in resultado:
    proximo = f"${r['proximo_total']} ({r['proximo_estado']})" if r['proximo_total'] else "ultimo"
    print(f"Cliente {r['id_cliente']}  Pedido {r['id']}  "
          f"${r['total']:8} [{r['estado']:10}]  siguiente: {proximo}")

--- Variacion entre pedidos por cliente ---
Cliente 1  Pedido 1  $1389.98  anterior:primero     variacion:-
Cliente 1  Pedido 3  $34.99  anterior:$1389.98    variacion:$-1354.99
Cliente 2  Pedido 2  $79.99  anterior:primero     variacion:-
Cliente 3  Pedido 4  $89.99  anterior:primero     variacion:-

--- Proximo pedido por cliente ---
Cliente 1  Pedido 1  $ 1389.98 [pagado    ]  siguiente: $34.99 (enviado)
Cliente 1  Pedido 3  $   34.99 [enviado   ]  siguiente: ultimo
Cliente 2  Pedido 2  $   79.99 [enviado   ]  siguiente: ultimo
Cliente 3  Pedido 4  $   89.99 [pagado    ]  siguiente: ultimo


### PARTITION BY sin ORDER BY

Cuando se usa `PARTITION BY` sin `ORDER BY` dentro del `OVER`,
la función opera sobre todo el grupo a la vez, no acumulativamente.

Útil para calcular totales o conteos de grupo y mostrarlos
junto a cada fila individual — algo que `GROUP BY` no permite.

```sql
COUNT(*) OVER (PARTITION BY columna)
-- devuelve el total del grupo para cada fila del grupo
```

In [50]:
# Total de pedidos y participacion porcentual de cada uno
print("--- Participacion de cada pedido en el total ---")
resultado = ejecutar_query("""
    SELECT
        id,
        id_cliente,
        total,
        estado,
        SUM(total) OVER ()                      AS total_general,
        ROUND(total * 100.0 / SUM(total) OVER (), 2) AS pct_del_total,
        COUNT(*) OVER (PARTITION BY id_cliente) AS pedidos_del_cliente
    FROM pedidos
    ORDER BY id_cliente, id;
""")
for r in resultado:
    print(f"Pedido {r['id']}  cliente {r['id_cliente']}  "
          f"${r['total']:8}  {r['pct_del_total']}% del total  "
          f"({r['pedidos_del_cliente']} pedidos del cliente)")

--- Participacion de cada pedido en el total ---
Pedido 1  cliente 1  $ 1389.98  87.15% del total  (2 pedidos del cliente)
Pedido 3  cliente 1  $   34.99  2.19% del total  (2 pedidos del cliente)
Pedido 2  cliente 2  $   79.99  5.02% del total  (1 pedidos del cliente)
Pedido 4  cliente 3  $   89.99  5.64% del total  (1 pedidos del cliente)


## 11. CTEs — Common Table Expressions

Una CTE es una consulta nombrada que se define antes de la query principal
y se puede referenciar como si fuera una tabla. Se declara con `WITH`.

```sql
WITH nombre_cte AS (
    SELECT ...
)
SELECT * FROM nombre_cte;
```

Ventajas sobre subconsultas:
- **Legibilidad** — el código se lee de arriba hacia abajo, como pasos
- **Reutilización** — la misma CTE se puede referenciar varias veces en la query
- **Debugging** — se puede ejecutar cada CTE por separado para verificar su resultado

Una CTE no es más rápida que una subconsulta equivalente — es una herramienta
de organización y legibilidad, no de rendimiento.

Tipos:
| Tipo | Descripción |
|------|-------------|
| CTE simple | Equivalente a una subconsulta nombrada |
| CTE múltiple | Varias CTEs encadenadas con comas |
| CTE recursiva | Se referencia a sí misma, útil para estructuras jerárquicas |

### CTE simple

El caso más común: dar nombre a una subconsulta para que la query
principal sea más legible.

Comparación de la misma lógica con subconsulta vs CTE:

```sql
-- Con subconsulta
SELECT * FROM (SELECT id, AVG(total) FROM pedidos GROUP BY id) sub WHERE ...

-- Con CTE
WITH promedios AS (SELECT id, AVG(total) FROM pedidos GROUP BY id)
SELECT * FROM promedios WHERE ...
```

La CTE es funcionalmente idéntica pero el código comunica mejor la intención.

In [51]:
# Clientes con su total gastado, solo los que superan el promedio general
print("--- Clientes sobre el promedio de gasto (CTE simple) ---")
resultado = ejecutar_query("""
    WITH gasto_por_cliente AS (
        SELECT
            id_cliente,
            COUNT(*)        AS total_pedidos,
            SUM(total)      AS total_gastado
        FROM pedidos
        GROUP BY id_cliente
    ),
    promedio_general AS (
        SELECT AVG(total_gastado) AS promedio
        FROM gasto_por_cliente
    )
    SELECT
        c.nombre            AS cliente,
        g.total_pedidos,
        g.total_gastado,
        ROUND(p.promedio, 2) AS promedio_general
    FROM gasto_por_cliente g
    JOIN clientes c ON g.id_cliente = c.id
    CROSS JOIN promedio_general p
    WHERE g.total_gastado > p.promedio
    ORDER BY g.total_gastado DESC;
""")
for r in resultado:
    print(f"{r['cliente']:20} {r['total_pedidos']} pedidos  "
          f"${r['total_gastado']}  (avg general: ${r['promedio_general']})")

--- Clientes sobre el promedio de gasto (CTE simple) ---
Ana Torres           2 pedidos  $1424.97  (avg general: $531.65)


### CTE múltiple — encadenar pasos

Se pueden definir varias CTEs separadas por coma.
Cada CTE puede referenciar a las anteriores.

Esto permite descomponer una query compleja en pasos claros,
como si fueran variables intermedias en código.

```sql
WITH paso1 AS (...),
     paso2 AS (SELECT * FROM paso1 WHERE ...),
     paso3 AS (SELECT * FROM paso2 JOIN otra ON ...)
SELECT * FROM paso3;
```

In [52]:
# Pipeline: detalle de pedidos → totales por cliente → ranking
print("--- Ranking de clientes por gasto (CTE multiple) ---")
resultado = ejecutar_query("""
    WITH detalle_completo AS (
        SELECT
            p.id_cliente,
            p.id            AS id_pedido,
            p.total,
            p.estado
        FROM pedidos p
        WHERE p.estado <> 'cancelado'
    ),
    resumen_cliente AS (
        SELECT
            id_cliente,
            COUNT(*)        AS pedidos,
            SUM(total)      AS gastado,
            MAX(total)      AS pedido_maximo
        FROM detalle_completo
        GROUP BY id_cliente
    ),
    ranking AS (
        SELECT
            id_cliente,
            pedidos,
            gastado,
            pedido_maximo,
            RANK() OVER (ORDER BY gastado DESC) AS posicion
        FROM resumen_cliente
    )
    SELECT
        r.posicion,
        c.nombre        AS cliente,
        r.pedidos,
        r.gastado,
        r.pedido_maximo
    FROM ranking r
    JOIN clientes c ON r.id_cliente = c.id
    ORDER BY r.posicion;
""")
for r in resultado:
    print(f"#{r['posicion']}  {r['cliente']:20} {r['pedidos']} pedidos  "
          f"total: ${r['gastado']}  max: ${r['pedido_maximo']}")

--- Ranking de clientes por gasto (CTE multiple) ---
#1  Ana Torres           2 pedidos  total: $1424.97  max: $1389.98
#2  Maria Lopez          1 pedidos  total: $89.99  max: $89.99
#3  Luis Gomez           1 pedidos  total: $79.99  max: $79.99


### CTE con INSERT, UPDATE y DELETE

Las CTEs no son solo para `SELECT` — también se pueden usar para
organizar operaciones de escritura complejas.

Un patrón muy útil es usar una CTE con `RETURNING` para capturar
las filas afectadas por un `UPDATE` o `DELETE` y procesarlas
en la query principal.

```sql
WITH filas_actualizadas AS (
    UPDATE tabla SET columna = valor
    WHERE condicion
    RETURNING *
)
SELECT * FROM filas_actualizadas;
```

In [53]:
# Actualizar estado de pedidos pendientes y obtener el detalle de los afectados
print("--- UPDATE con CTE y RETURNING ---")
resultado = ejecutar_query("""
    WITH pedidos_actualizados AS (
        UPDATE pedidos
        SET estado = 'pagado'
        WHERE estado = 'pendiente'
        RETURNING id, id_cliente, total, estado
    )
    SELECT
        pu.id           AS pedido_id,
        c.nombre        AS cliente,
        pu.total,
        pu.estado       AS estado_nuevo
    FROM pedidos_actualizados pu
    JOIN clientes c ON pu.id_cliente = c.id;
""")
if resultado:
    for r in resultado:
        print(f"Pedido {r['pedido_id']}  {r['cliente']:20} ${r['total']}  → {r['estado_nuevo']}")
else:
    print("No habia pedidos pendientes")

--- UPDATE con CTE y RETURNING ---
No habia pedidos pendientes


### CTE recursiva

Una CTE recursiva se referencia a sí misma. Tiene dos partes:

```sql
WITH RECURSIVE nombre AS (
    -- Caso base: punto de partida, sin recursion
    SELECT ...

    UNION ALL

    -- Caso recursivo: se une con la CTE misma
    SELECT ... FROM nombre WHERE condicion_de_corte
)
SELECT * FROM nombre;
```

PostgreSQL ejecuta el caso base una vez, luego aplica el caso recursivo
repetidamente sobre el resultado anterior hasta que no hay más filas nuevas.

Casos de uso típicos:
- Recorrer jerarquías (categorías con subcategorías, organigramas)
- Generar series de números o fechas
- Calcular caminos en grafos

In [54]:
# Recorrer la jerarquia de categorias (padre → hijo)
print("--- Jerarquia de categorias (CTE recursiva) ---")
resultado = ejecutar_query("""
    WITH RECURSIVE jerarquia AS (
        SELECT
            id,
            nombre,
            id_padre,
            0                       AS nivel,
            nombre::TEXT            AS camino
        FROM categorias
        WHERE id_padre IS NULL

        UNION ALL

        SELECT
            c.id,
            c.nombre,
            c.id_padre,
            j.nivel + 1,
            (j.camino || ' > ' || c.nombre)::TEXT
        FROM categorias c
        JOIN jerarquia j ON c.id_padre = j.id
    )
    SELECT
        nivel,
        REPEAT('  ', nivel) || nombre    AS nombre_indentado,
        camino
    FROM jerarquia
    ORDER BY camino;
""")
for r in resultado:
    print(f"Nivel {r['nivel']}  {r['nombre_indentado']:25} {r['camino']}")

--- Jerarquia de categorias (CTE recursiva) ---
Nivel 0  Hogar                     Hogar
Nivel 1    Deportes                Hogar > Deportes
Nivel 1    Electronica             Hogar > Electronica
Nivel 0  Ropa                      Ropa


In [55]:
# Generar una serie de fechas con CTE recursiva
print("--- Serie de fechas (CTE recursiva) ---")
resultado = ejecutar_query("""
    WITH RECURSIVE serie_fechas AS (
        SELECT CURRENT_DATE - INTERVAL '6 days' AS fecha

        UNION ALL

        SELECT fecha + INTERVAL '1 day'
        FROM serie_fechas
        WHERE fecha < CURRENT_DATE
    )
    SELECT
        fecha::DATE,
        TO_CHAR(fecha, 'Day') AS dia_semana
    FROM serie_fechas
    ORDER BY fecha;
""")
for r in resultado:
    print(f"{r['fecha']}  {r['dia_semana'].strip()}")

--- Serie de fechas (CTE recursiva) ---
2026-06-09  Tuesday
2026-06-10  Wednesday
2026-06-11  Thursday
2026-06-12  Friday
2026-06-13  Saturday
2026-06-14  Sunday
2026-06-15  Monday


## 12. Transacciones ACID

Una transacción es un conjunto de operaciones SQL que se ejecutan
como una unidad indivisible: o todas tienen éxito o ninguna se aplica.

**ACID** es el acrónimo de las cuatro propiedades que garantiza una transacción:

| Propiedad | Nombre | Descripción |
|-----------|--------|-------------|
| **A** | Atomicity (Atomicidad) | Todo o nada — si algo falla, se revierten todos los cambios |
| **C** | Consistency (Consistencia) | La base pasa de un estado válido a otro estado válido |
| **I** | Isolation (Aislamiento) | Las transacciones concurrentes no se interfieren entre sí |
| **D** | Durability (Durabilidad) | Una vez confirmada, la transacción persiste aunque el sistema falle |

En psycopg, `autocommit=False` (el default) abre una transacción implícita
con cada sentencia. Hay que cerrarla explícitamente con `commit()` o `rollback()`.

En nuestro notebook usamos `autocommit=True` para simplificar las consultas.
Para trabajar con transacciones, abrimos conexiones con `autocommit=False`.

### COMMIT y ROLLBACK

```sql
BEGIN;        -- inicia la transaccion explicitamente
  INSERT ...
  UPDATE ...
COMMIT;       -- confirma todos los cambios

-- o si algo sale mal:
ROLLBACK;     -- revierte todos los cambios desde el BEGIN
```

En psycopg el `BEGIN` es implícito — al abrir una conexión con
`autocommit=False` ya estamos dentro de una transacción.
Solo necesitamos llamar a `commit()` o `rollback()`.

`COMMIT` — persiste todos los cambios en la base.
`ROLLBACK` — descarta todos los cambios desde el inicio de la transacción.

In [56]:
# Transaccion exitosa: transferir stock entre productos
print("--- Transaccion exitosa ---")
conn = psycopg.connect(CONN_STR, autocommit=False, row_factory=dict_row)
try:
    with conn.cursor() as cur:
        # Verificar stock inicial
        cur.execute("SELECT nombre, stock FROM productos WHERE id IN (1, 2);")
        for r in cur.fetchall():
            print(f"  Inicial  — {r['nombre']:20} stock: {r['stock']}")

        # Descontar stock del producto 1
        cur.execute("""
            UPDATE productos SET stock = stock - 2
            WHERE id = 1 AND stock >= 2
            RETURNING nombre, stock;
        """)
        r = cur.fetchone()
        print(f"  Restado  — {r['nombre']:20} stock: {r['stock']}")

        # Agregar stock al producto 2
        cur.execute("""
            UPDATE productos SET stock = stock + 2
            WHERE id = 2
            RETURNING nombre, stock;
        """)
        r = cur.fetchone()
        print(f"  Sumado   — {r['nombre']:20} stock: {r['stock']}")

    conn.commit()
    print("  COMMIT ejecutado — cambios persistidos")

except Exception as e:
    conn.rollback()
    print(f"  ROLLBACK — error: {e}")

finally:
    conn.close()

--- Transaccion exitosa ---
  Inicial  — Auriculares          stock: 35
  Inicial  — Laptop 15"           stock: 10
  Restado  — Laptop 15"           stock: 8
  Sumado   — Auriculares          stock: 37
  COMMIT ejecutado — cambios persistidos


### ROLLBACK automático ante un error

El valor real de una transacción se ve cuando algo falla a mitad de camino.
Sin transacción, los cambios anteriores al error quedan persistidos
dejando la base en un estado inconsistente.

Con transacción, un error en cualquier punto revierte todo lo que
se hizo desde el `BEGIN`, dejando la base exactamente como estaba.

In [57]:
# Transaccion que falla a mitad: el segundo UPDATE viola un constraint
print("--- Transaccion con error y ROLLBACK ---")

# Estado antes
antes = ejecutar_query("SELECT nombre, stock FROM productos WHERE id IN (1, 2);")
print("Antes:")
for r in antes:
    print(f"  {r['nombre']:20} stock: {r['stock']}")

conn = psycopg.connect(CONN_STR, autocommit=False, row_factory=dict_row)
try:
    with conn.cursor() as cur:
        # Primera operacion: ok
        cur.execute("""
            UPDATE productos SET stock = stock - 1
            WHERE id = 1
            RETURNING nombre, stock;
        """)
        r = cur.fetchone()
        print(f"\n  UPDATE 1 ok — {r['nombre']} stock: {r['stock']}")

        # Segunda operacion: stock negativo viola el CHECK (si existiera)
        # Simulamos el error insertando un total negativo en pedidos
        cur.execute("""
            INSERT INTO pedidos (id_cliente, total, estado)
            VALUES (1, -999.99, 'pagado');
        """)

    conn.commit()
    print("  COMMIT")

except Exception as e:
    conn.rollback()
    print(f"\n  ERROR: {e}")
    print("  ROLLBACK ejecutado — ningun cambio persistido")

finally:
    conn.close()

# Estado despues: debe ser identico al estado antes
despues = ejecutar_query("SELECT nombre, stock FROM productos WHERE id IN (1, 2);")
print("\nDespues del ROLLBACK:")
for r in despues:
    print(f"  {r['nombre']:20} stock: {r['stock']}")

--- Transaccion con error y ROLLBACK ---
Antes:
  Laptop 15"           stock: 8
  Auriculares          stock: 37

  UPDATE 1 ok — Laptop 15" stock: 7

  ERROR: new row for relation "pedidos" violates check constraint "pedidos_total_check"
DETAIL:  Failing row contains (5, 1, -999.99, pagado, 2026-06-15 05:00:46.821767).
  ROLLBACK ejecutado — ningun cambio persistido

Despues del ROLLBACK:
  Laptop 15"           stock: 8
  Auriculares          stock: 37


### SAVEPOINT — puntos de guardado parciales

Un `SAVEPOINT` marca un punto dentro de una transacción al que se puede
volver sin descartar toda la transacción.

```sql
BEGIN;
  INSERT ...          -- operacion 1
  SAVEPOINT sp1;      -- marca el punto
  UPDATE ...          -- operacion 2
  ROLLBACK TO sp1;    -- deshace solo la operacion 2
  INSERT ...          -- operacion 3
COMMIT;               -- persiste operaciones 1 y 3
```

Útil cuando una transacción larga tiene pasos opcionales o de riesgo
que queremos poder deshacer sin perder todo el trabajo anterior.

In [58]:
print("--- SAVEPOINT ---")

conn = psycopg.connect(CONN_STR, autocommit=False, row_factory=dict_row)
try:
    with conn.cursor() as cur:

        # Operacion 1: nuevo cliente
        cur.execute("""
            INSERT INTO clientes (nombre, email)
            VALUES ('Cliente Savepoint', 'savepoint@email.com')
            RETURNING id, nombre;
        """)
        cliente = cur.fetchone()
        print(f"  INSERT cliente ok — id: {cliente['id']} nombre: {cliente['nombre']}")

        # Marcar savepoint antes de una operacion riesgosa
        cur.execute("SAVEPOINT sp1;")
        print("  SAVEPOINT sp1 creado")

        # Operacion 2 riesgosa: pedido con total invalido
        try:
            cur.execute("""
                INSERT INTO pedidos (id_cliente, total, estado)
                VALUES (%s, -50.00, 'pagado');
            """, (cliente['id'],))
        except Exception as e:
            print(f"  ERROR en operacion 2: {e}")
            cur.execute("ROLLBACK TO sp1;")
            print("  ROLLBACK TO sp1 — volvemos al savepoint")

        # Operacion 3: pedido valido
        cur.execute("""
            INSERT INTO pedidos (id_cliente, total, estado)
            VALUES (%s, 150.00, 'pendiente')
            RETURNING id, total, estado;
        """, (cliente['id'],))
        pedido = cur.fetchone()
        print(f"  INSERT pedido ok — id: {pedido['id']} total: ${pedido['total']}")

    conn.commit()
    print("  COMMIT — cliente y pedido valido persistidos")

except Exception as e:
    conn.rollback()
    print(f"  ROLLBACK total — error: {e}")

finally:
    conn.close()

# Verificar resultado final
print("\nVerificacion:")
resultado = ejecutar_query("""
    SELECT c.nombre AS cliente, p.total, p.estado
    FROM clientes c
    JOIN pedidos p ON p.id_cliente = c.id
    WHERE c.email = 'savepoint@email.com';
""")
for r in resultado:
    print(f"  {r['cliente']:20} ${r['total']}  {r['estado']}")

--- SAVEPOINT ---
  INSERT cliente ok — id: 7 nombre: Cliente Savepoint
  SAVEPOINT sp1 creado
  ERROR en operacion 2: new row for relation "pedidos" violates check constraint "pedidos_total_check"
DETAIL:  Failing row contains (6, 7, -50.00, pagado, 2026-06-15 05:00:47.128428).
  ROLLBACK TO sp1 — volvemos al savepoint
  INSERT pedido ok — id: 7 total: $150.00
  COMMIT — cliente y pedido valido persistidos

Verificacion:
  Cliente Savepoint    $150.00  pendiente


### Niveles de aislamiento

El aislamiento de las transacciones concurrentes tiene cuatro niveles.
A mayor aislamiento, mayor consistencia pero menor concurrencia.

| Nivel | Dirty Read | Non-repeatable Read | Phantom Read |
|-------|-----------|---------------------|--------------|
| `READ UNCOMMITTED` | posible | posible | posible |
| `READ COMMITTED` | ✓ protegido | posible | posible |
| `REPEATABLE READ` | ✓ protegido | ✓ protegido | posible |
| `SERIALIZABLE` | ✓ protegido | ✓ protegido | ✓ protegido |

- **Dirty Read** — leer cambios de otra transacción aún no confirmada
- **Non-repeatable Read** — leer el mismo dato dos veces y obtener valores distintos
- **Phantom Read** — una segunda lectura devuelve filas que no existían antes

PostgreSQL usa `READ COMMITTED` por default y no implementa `READ UNCOMMITTED`
(lo trata igual que `READ COMMITTED`). Para máxima consistencia: `SERIALIZABLE`.

In [59]:
# Verificar y cambiar nivel de aislamiento
print("--- Nivel de aislamiento ---")

conn = psycopg.connect(CONN_STR, autocommit=False, row_factory=dict_row)
try:
    with conn.cursor() as cur:

        # Ver nivel actual
        cur.execute("SHOW transaction_isolation;")
        nivel = cur.fetchone()
        print(f"  Nivel actual: {nivel['transaction_isolation']}")

        # Cambiar a REPEATABLE READ para esta transaccion
        cur.execute("SET TRANSACTION ISOLATION LEVEL REPEATABLE READ;")

        cur.execute("SHOW transaction_isolation;")
        nivel = cur.fetchone()
        print(f"  Nivel nuevo:  {nivel['transaction_isolation']}")

        # Operacion dentro del nivel REPEATABLE READ
        cur.execute("SELECT COUNT(*) AS total FROM pedidos;")
        r = cur.fetchone()
        print(f"  Pedidos al inicio de la transaccion: {r['total']}")

    conn.commit()

finally:
    conn.close()

--- Nivel de aislamiento ---
  Nivel actual: read committed
  Nivel nuevo:  repeatable read
  Pedidos al inicio de la transaccion: 5


## 13. Vistas

Una vista es una query guardada con nombre en la base de datos.
Se usa como si fuera una tabla, pero no almacena datos — cada vez
que se consulta ejecuta la query subyacente.

```sql
CREATE VIEW nombre_vista AS
SELECT ...;
```

Ventajas:
- **Simplicidad** — encapsulan queries complejas detrás de un nombre simple
- **Reutilización** — la misma lógica disponible para múltiples queries
- **Seguridad** — se puede dar acceso a una vista sin exponer las tablas base
- **Mantenimiento** — si cambia la lógica, se actualiza en un solo lugar

Limitaciones:
- No almacenan datos — cada consulta re-ejecuta la query completa
- No se pueden indexar directamente (para eso existen las vistas materializadas)
- En PostgreSQL, una vista simple no acepta `INSERT`/`UPDATE`/`DELETE`
  a menos que sea una vista simple de una sola tabla o se defina un trigger

### CREATE VIEW

La vista queda guardada en la base de datos y persiste entre sesiones.
Se consulta exactamente igual que una tabla.

`CREATE OR REPLACE VIEW` permite actualizar la definición de una vista
existente sin necesidad de borrarla y recrearla — siempre que las
columnas existentes no cambien de nombre ni de tipo.

In [60]:
# Vista: resumen de pedidos con datos del cliente
with obtener_conexion() as conn:
    with conn.cursor() as cur:
        cur.execute("""
            CREATE OR REPLACE VIEW vista_pedidos_completos AS
            SELECT
                p.id                AS pedido_id,
                c.nombre            AS cliente,
                c.email,
                p.total,
                p.estado,
                p.creado_en::DATE   AS fecha
            FROM pedidos p
            JOIN clientes c ON p.id_cliente = c.id;
        """)

print("Vista vista_pedidos_completos creada")

# Consultar la vista como si fuera una tabla
print("\n--- Consultando la vista ---")
resultado = ejecutar_query("""
    SELECT * FROM vista_pedidos_completos
    ORDER BY fecha, pedido_id;
""")
for r in resultado:
    print(f"Pedido {r['pedido_id']}  {r['cliente']:20} ${r['total']:8}  "
          f"{r['estado']:12} {r['fecha']}")

Vista vista_pedidos_completos creada

--- Consultando la vista ---
Pedido 1  Ana Torres           $ 1389.98  pagado       2026-06-15
Pedido 2  Luis Gomez           $   79.99  enviado      2026-06-15
Pedido 3  Ana Torres           $   34.99  enviado      2026-06-15
Pedido 4  Maria Lopez          $   89.99  pagado       2026-06-15
Pedido 7  Cliente Savepoint    $  150.00  pendiente    2026-06-15


### Vistas para análisis frecuentes

Un caso de uso muy común es encapsular queries analíticas que
se ejecutan regularmente — reportes de ventas, métricas de clientes,
stock crítico, etc.

En lugar de reescribir la query cada vez, se define una vista
y se consulta directamente, pudiendo además filtrarla con `WHERE`
o combinarla con otras tablas o vistas.

In [61]:
with obtener_conexion() as conn:
    with conn.cursor() as cur:

        # Vista: metricas por cliente
        cur.execute("""
            CREATE OR REPLACE VIEW vista_metricas_clientes AS
            SELECT
                c.id                        AS id_cliente,
                c.nombre                    AS cliente,
                c.email,
                COUNT(p.id)                 AS total_pedidos,
                COALESCE(SUM(p.total), 0)   AS total_gastado,
                COALESCE(ROUND(AVG(p.total), 2), 0) AS ticket_promedio,
                MAX(p.creado_en::DATE)       AS ultimo_pedido
            FROM clientes c
            LEFT JOIN pedidos p ON p.id_cliente = c.id
            GROUP BY c.id, c.nombre, c.email;
        """)

        # Vista: productos con datos de categoria y stock
        cur.execute("""
            CREATE OR REPLACE VIEW vista_catalogo AS
            SELECT
                p.id                AS id_producto,
                p.nombre            AS producto,
                c.nombre            AS categoria,
                p.precio,
                p.stock,
                CASE
                    WHEN p.stock = 0    THEN 'sin stock'
                    WHEN p.stock < 10   THEN 'stock bajo'
                    ELSE                     'disponible'
                END                 AS estado_stock
            FROM productos p
            LEFT JOIN categorias c ON p.id_categoria = c.id;
        """)

print("Vistas de analisis creadas")

# Consultar metricas de clientes
print("\n--- Metricas por cliente ---")
resultado = ejecutar_query("""
    SELECT * FROM vista_metricas_clientes
    ORDER BY total_gastado DESC;
""")
for r in resultado:
    print(f"{r['cliente']:20} {r['total_pedidos']} pedidos  "
          f"${r['total_gastado']:8}  avg: ${r['ticket_promedio']}")

print()

# Consultar catalogo filtrando por estado de stock
print("--- Productos disponibles ---")
resultado = ejecutar_query("""
    SELECT producto, categoria, precio, stock, estado_stock
    FROM vista_catalogo
    WHERE estado_stock = 'disponible'
    ORDER BY categoria, precio;
""")
for r in resultado:
    print(f"{r['categoria']:15} {r['producto']:22} ${r['precio']:8} stock:{r['stock']}")

Vistas de analisis creadas

--- Metricas por cliente ---
Ana Torres           2 pedidos  $ 1424.97  avg: $712.49
Cliente Savepoint    1 pedidos  $  150.00  avg: $150.00
Maria Lopez          1 pedidos  $   89.99  avg: $89.99
Luis Gomez           1 pedidos  $   79.99  avg: $79.99
Carlos Ruiz          0 pedidos  $       0  avg: $0
Sin Email            0 pedidos  $       0  avg: $0

--- Productos disponibles ---
Deportes        Zapatillas 42          $   79.99 stock:50
Electronica     Auriculares Pro        $   89.99 stock:20
Electronica     Auriculares            $   89.99 stock:37
Hogar           Lampara LED            $   34.99 stock:60
Ropa            Camiseta M             $   19.99 stock:100


### Filtrar y combinar vistas

Las vistas se pueden usar en cualquier lugar donde se usa una tabla:
con `WHERE`, `ORDER BY`, `JOIN`, dentro de CTEs, etc.

Esto permite construir capas de abstracción: una vista base con los datos
crudos, y otras vistas o queries encima que la filtran o combinan.

In [62]:
# Filtrar la vista por condicion
print("--- Clientes sin pedidos (desde la vista) ---")
resultado = ejecutar_query("""
    SELECT cliente, email, total_pedidos
    FROM vista_metricas_clientes
    WHERE total_pedidos = 0;
""")
for r in resultado:
    email = r['email'] if r['email'] else 'sin email'
    print(f"{r['cliente']:20} {email}")

print()

# Combinar dos vistas
print("--- Pedidos de productos con stock bajo ---")
resultado = ejecutar_query("""
    SELECT
        vpc.cliente,
        vpc.pedido_id,
        vpc.total,
        vpc.estado
    FROM vista_pedidos_completos vpc
    WHERE vpc.pedido_id IN (
        SELECT DISTINCT pd.id_pedido
        FROM pedido_detalle pd
        JOIN vista_catalogo vc ON pd.id_producto = vc.id_producto
        WHERE vc.estado_stock = 'stock bajo'
    )
    ORDER BY vpc.pedido_id;
""")
for r in resultado:
    print(f"{r['cliente']:20} Pedido {r['pedido_id']}  ${r['total']}  {r['estado']}")

--- Clientes sin pedidos (desde la vista) ---
Carlos Ruiz          carlos@email.com
Sin Email            sin email

--- Pedidos de productos con stock bajo ---
Ana Torres           Pedido 1  $1389.98  pagado


### Listar, inspeccionar y eliminar vistas

`information_schema.views` lista todas las vistas del schema
junto con la query que las define.

`DROP VIEW` elimina la vista — solo la definición, nunca los datos,
ya que la vista no almacena nada.

`DROP VIEW IF EXISTS ... CASCADE` elimina además cualquier objeto
que dependa de esa vista (otras vistas que la referencien).

In [63]:
# Listar vistas del schema
print("--- Vistas en el schema ---")
resultado = ejecutar_query("""
    SELECT
        table_name      AS vista,
        view_definition AS definicion
    FROM information_schema.views
    WHERE table_schema = 'public'
    ORDER BY table_name;
""")
for r in resultado:
    # Mostrar solo la primera linea de la definicion para no saturar la salida
    primera_linea = r['definicion'].strip().splitlines()[0]
    print(f"{r['vista']:35} {primera_linea}")

print()

# Eliminar una vista
with obtener_conexion() as conn:
    with conn.cursor() as cur:
        cur.execute("DROP VIEW IF EXISTS vista_catalogo CASCADE;")
        cur.execute("DROP VIEW IF EXISTS vista_metricas_clientes CASCADE;")
        cur.execute("DROP VIEW IF EXISTS vista_pedidos_completos CASCADE;")

print("Vistas eliminadas")

--- Vistas en el schema ---
vista_catalogo                      SELECT p.id AS id_producto,
vista_metricas_clientes             SELECT c.id AS id_cliente,
vista_pedidos_completos             SELECT p.id AS pedido_id,

Vistas eliminadas


## 14. CRUD en Python con psycopg

En las secciones anteriores usamos `ejecutar_query()` como atajo genérico.
En esta sección vemos cómo estructurar el acceso a datos en Python de forma
profesional: funciones específicas por operación, parámetros seguros,
manejo de errores y patrones reutilizables.

Conceptos clave de psycopg v3 que usamos en esta sección:

| Concepto | Descripción |
|----------|-------------|
| Parámetros `%s` | Placeholders para valores — psycopg los escapa automáticamente |
| `execute(sql, params)` | Ejecuta con parámetros posicionales como tupla |
| `executemany(sql, seq)` | Ejecuta la misma query para una lista de parámetros |
| `fetchone()` | Recupera una sola fila |
| `fetchall()` | Recupera todas las filas |
| `rowcount` | Cantidad de filas afectadas por el último `execute` |
| `RETURNING` | Cláusula de PostgreSQL que devuelve las filas afectadas |

El uso de parámetros `%s` es obligatorio para valores dinámicos.
Nunca se concatena un valor directamente en el string SQL — eso
abre vulnerabilidades de SQL injection.

```python
# MAL — vulnerable a SQL injection
cur.execute(f"SELECT * FROM clientes WHERE email = '{email}'")

# BIEN — psycopg escapa el valor
cur.execute("SELECT * FROM clientes WHERE email = %s", (email,))
```

### Estructura recomendada

En una aplicación real el acceso a datos se organiza en funciones
o clases separadas por entidad — una por tabla o dominio.

Cada función tiene una responsabilidad única:
- Recibe los datos como parámetros Python
- Construye y ejecuta la query con parámetros seguros
- Devuelve el resultado como diccionario o lista de diccionarios
- Maneja sus propios errores

Esto separa la lógica SQL del resto de la aplicación y facilita
el testing y el mantenimiento.

In [64]:
# Funciones CRUD para la entidad clientes

def crear_cliente(nombre, email=None):
    """Inserta un cliente y devuelve el registro creado."""
    with obtener_conexion() as conn:
        with conn.cursor() as cur:
            cur.execute("""
                INSERT INTO clientes (nombre, email)
                VALUES (%s, %s)
                RETURNING id, nombre, email, creado_en;
            """, (nombre, email))
            return cur.fetchone()

def obtener_cliente(id_cliente):
    """Devuelve un cliente por ID o None si no existe."""
    with obtener_conexion() as conn:
        with conn.cursor() as cur:
            cur.execute("""
                SELECT id, nombre, email, creado_en
                FROM clientes
                WHERE id = %s;
            """, (id_cliente,))
            return cur.fetchone()

def listar_clientes(solo_con_email=False):
    """Devuelve todos los clientes, opcionalmente solo los que tienen email."""
    filtro = "WHERE email IS NOT NULL" if solo_con_email else ""
    with obtener_conexion() as conn:
        with conn.cursor() as cur:
            cur.execute(f"""
                SELECT id, nombre, email, creado_en
                FROM clientes
                {filtro}
                ORDER BY nombre;
            """)
            return cur.fetchall()

def actualizar_cliente(id_cliente, nombre=None, email=None):
    """Actualiza nombre y/o email. Devuelve el registro actualizado o None."""
    if nombre is None and email is None:
        raise ValueError("Debe especificar al menos un campo a actualizar")

    campos = []
    valores = []
    if nombre is not None:
        campos.append("nombre = %s")
        valores.append(nombre)
    if email is not None:
        campos.append("email = %s")
        valores.append(email)
    valores.append(id_cliente)

    with obtener_conexion() as conn:
        with conn.cursor() as cur:
            cur.execute(f"""
                UPDATE clientes
                SET {', '.join(campos)}
                WHERE id = %s
                RETURNING id, nombre, email;
            """, valores)
            return cur.fetchone()

def eliminar_cliente(id_cliente):
    """Elimina un cliente. Devuelve True si existia, False si no."""
    with obtener_conexion() as conn:
        with conn.cursor() as cur:
            cur.execute("""
                DELETE FROM clientes
                WHERE id = %s
                RETURNING id;
            """, (id_cliente,))
            return cur.fetchone() is not None

### Usando las funciones CRUD

Las funciones encapsulan completamente el SQL — el código que las llama
no necesita saber nada sobre la base de datos, solo qué parámetros pasar
y qué esperar como resultado.

In [65]:
# CREATE
nuevo = crear_cliente("Roberto Silva", "roberto@email.com")
print(f"Creado  — id:{nuevo['id']} nombre:{nuevo['nombre']} email:{nuevo['email']}")

# READ uno
cliente = obtener_cliente(nuevo['id'])
print(f"Leido   — id:{cliente['id']} nombre:{cliente['nombre']}")

# READ lista
print("\nTodos los clientes con email:")
for c in listar_clientes(solo_con_email=True):
    print(f"  {c['id']}  {c['nombre']:20} {c['email']}")

# UPDATE
actualizado = actualizar_cliente(nuevo['id'], email="roberto.nuevo@email.com")
print(f"\nActualizado — email: {actualizado['email']}")

# DELETE
eliminado = eliminar_cliente(nuevo['id'])
print(f"Eliminado — resultado: {eliminado}")

# Verificar que ya no existe
cliente = obtener_cliente(nuevo['id'])
print(f"Buscar eliminado — resultado: {cliente}")

Creado  — id:8 nombre:Roberto Silva email:roberto@email.com
Leido   — id:8 nombre:Roberto Silva

Todos los clientes con email:
  1  Ana Torres           ana@email.com
  4  Carlos Ruiz          carlos@email.com
  7  Cliente Savepoint    savepoint@email.com
  2  Luis Gomez           luis@email.com
  3  Maria Lopez          maria@email.com
  8  Roberto Silva        roberto@email.com

Actualizado — email: roberto.nuevo@email.com
Eliminado — resultado: True
Buscar eliminado — resultado: None


### CRUD con validaciones de negocio

En una entidad con más restricciones — como `productos`, que tiene precio,
stock y categoría — las funciones pueden incluir validaciones de negocio
antes de ejecutar la query, devolviendo errores descriptivos.

`executemany` es la forma eficiente de insertar múltiples filas:
ejecuta la misma query preparada una vez por cada elemento de la lista,
siendo significativamente más rápido que llamar a `execute` en un loop.

In [66]:
# CRUD para productos con validaciones

def crear_producto(nombre, precio, stock=0, id_categoria=None):
    """Crea un producto con validaciones de negocio."""
    if precio < 0:
        raise ValueError(f"El precio no puede ser negativo: {precio}")
    if stock < 0:
        raise ValueError(f"El stock no puede ser negativo: {stock}")

    with obtener_conexion() as conn:
        with conn.cursor() as cur:
            cur.execute("""
                INSERT INTO productos (nombre, precio, stock, id_categoria)
                VALUES (%s, %s, %s, %s)
                RETURNING id, nombre, precio, stock;
            """, (nombre, precio, stock, id_categoria))
            return cur.fetchone()

def crear_productos_lote(lista_productos):
    """
    Inserta múltiples productos eficientemente.
    lista_productos: lista de tuplas (nombre, precio, stock, id_categoria)
    """
    with obtener_conexion() as conn:
        with conn.cursor() as cur:
            cur.executemany("""
                INSERT INTO productos (nombre, precio, stock, id_categoria)
                VALUES (%s, %s, %s, %s);
            """, lista_productos)
            return cur.rowcount

def ajustar_stock(id_producto, cantidad):
    """
    Ajusta el stock sumando o restando cantidad.
    cantidad positiva = entrada, negativa = salida.
    Devuelve el stock resultante o None si no hay suficiente stock.
    """
    with obtener_conexion() as conn:
        with conn.cursor() as cur:
            cur.execute("""
                UPDATE productos
                SET stock = stock + %s
                WHERE id = %s
                AND stock + %s >= 0
                RETURNING id, nombre, stock;
            """, (cantidad, id_producto, cantidad))
            return cur.fetchone()

def buscar_productos(texto=None, precio_min=None, precio_max=None, id_categoria=None):
    """Busqueda flexible con filtros opcionales."""
    condiciones = []
    valores = []

    if texto:
        condiciones.append("nombre ILIKE %s")
        valores.append(f"%{texto}%")
    if precio_min is not None:
        condiciones.append("precio >= %s")
        valores.append(precio_min)
    if precio_max is not None:
        condiciones.append("precio <= %s")
        valores.append(precio_max)
    if id_categoria is not None:
        condiciones.append("id_categoria = %s")
        valores.append(id_categoria)

    where = "WHERE " + " AND ".join(condiciones) if condiciones else ""

    with obtener_conexion() as conn:
        with conn.cursor() as cur:
            cur.execute(f"""
                SELECT id, nombre, precio, stock, id_categoria
                FROM productos
                {where}
                ORDER BY precio;
            """, valores)
            return cur.fetchall()

In [67]:
# Probar las funciones de productos
nuevo = crear_producto("Monitor 4K", 349.99, stock=15, id_categoria=1)
print(f"Creado — {nuevo['nombre']} ${nuevo['precio']} stock:{nuevo['stock']}")

# Insertar en lote
cantidad = crear_productos_lote([
    ("Teclado Mecanico", 129.99, 25, 1),
    ("Mouse Inalambrico", 49.99, 40, 1),
])
print(f"Lote — {cantidad} productos insertados")

# Ajustar stock
resultado = ajustar_stock(nuevo['id'], -3)
print(f"Stock ajustado — {resultado['nombre']} stock: {resultado['stock']}")

# Intento de stock negativo
resultado = ajustar_stock(nuevo['id'], -999)
print(f"Stock insuficiente — resultado: {resultado}")

# Busqueda flexible
print("\n--- Busqueda: electronica entre $40 y $150 ---")
for p in buscar_productos(precio_min=40, precio_max=150, id_categoria=1):
    print(f"  {p['nombre']:22} ${p['precio']}  stock:{p['stock']}")

print("\n--- Busqueda por texto: 'tec' ---")
for p in buscar_productos(texto="tec"):
    print(f"  {p['nombre']:22} ${p['precio']}")

Creado — Monitor 4K $349.99 stock:15
Lote — 2 productos insertados
Stock ajustado — Monitor 4K stock: 12
Stock insuficiente — resultado: None

--- Busqueda: electronica entre $40 y $150 ---
  Mouse Inalambrico      $49.99  stock:40
  Auriculares Pro        $89.99  stock:20
  Auriculares            $89.99  stock:37
  Teclado Mecanico       $129.99  stock:25

--- Busqueda por texto: 'tec' ---
  Teclado Mecanico       $129.99


### Manejo de errores específicos de psycopg

psycopg mapea los errores de PostgreSQL a excepciones Python específicas,
todas heredando de `psycopg.Error`.

Las más útiles:

| Excepción | Cuándo ocurre |
|-----------|---------------|
| `psycopg.errors.UniqueViolation` | Se viola un constraint `UNIQUE` o `PRIMARY KEY` |
| `psycopg.errors.ForeignKeyViolation` | Se viola una `FOREIGN KEY` |
| `psycopg.errors.CheckViolation` | Se viola un constraint `CHECK` |
| `psycopg.errors.NotNullViolation` | Se intenta insertar `NULL` en columna `NOT NULL` |

Capturar errores específicos permite dar mensajes descriptivos
al usuario en lugar de exponer el error interno de PostgreSQL.

In [68]:
import psycopg.errors

def crear_cliente_seguro(nombre, email):
    """Crear cliente con manejo de errores especificos."""
    try:
        return crear_cliente(nombre, email), None
    except psycopg.errors.UniqueViolation:
        return None, f"El email '{email}' ya esta registrado"
    except psycopg.errors.NotNullViolation:
        return None, "El nombre es obligatorio"
    except psycopg.Error as e:
        return None, f"Error de base de datos: {e}"

# Email duplicado
cliente, error = crear_cliente_seguro("Otro Usuario", "ana@email.com")
print(f"Email duplicado — cliente: {cliente}  error: {error}")

# Insercion valida
cliente, error = crear_cliente_seguro("Usuario Nuevo", "nuevo@email.com")
print(f"Valido — id: {cliente['id']} nombre: {cliente['nombre']}  error: {error}")

# Nombre nulo
cliente, error = crear_cliente_seguro(None, "sin_nombre@email.com")
print(f"Nombre nulo — cliente: {cliente}  error: {error}")

# Limpiar
eliminar_cliente(cliente['id']) if not error else None

Email duplicado — cliente: None  error: El email 'ana@email.com' ya esta registrado
Valido — id: 10 nombre: Usuario Nuevo  error: None
Nombre nulo — cliente: None  error: El nombre es obligatorio
